In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split, GridSearchCV, 
    KFold, RepeatedKFold
)
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, 
    mean_absolute_percentage_error
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.inspection import permutation_importance
import warnings
import os
import joblib
import json
import sys
import platform
from scipy.stats import pearsonr, spearmanr, sem, t
from scipy import stats
import matplotlib.pyplot as plt
import logging
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
import time

# ============================================================================
# CONFIGURATION
# ============================================================================
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ============================================================================
# REPRODUCIBILITY REPORT
# ============================================================================
def generate_reproducibility_report():
    """Generate comprehensive reproducibility report."""
    report = {
        'timestamp': datetime.now().isoformat(),
        'python_version': sys.version,
        'platform': platform.platform(),
        'packages': {},
        'random_seeds': {
            'main': 42,
            'grid_search': 42,
            'cv': 42,
            'train_test_split': 42,
            'repeated_cv': 42,
            'bootstrap': 42,
            'permutation_importance': 42
        }
    }
    
    packages = ['numpy', 'pandas', 'sklearn', 'scipy', 'matplotlib', 'joblib']
    
    for pkg in packages:
        try:
            module = __import__(pkg)
            report['packages'][pkg] = module.__version__
        except:
            report['packages'][pkg] = 'not installed'
    
    return report

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def sanitize_feature_names(df: pd.DataFrame) -> pd.DataFrame:
    """Sanitize feature names to remove special characters."""
    import re
    def clean_name(name: str) -> str:
        name = re.sub(r'[()\[\]{}<>/\\|;:,.\'"]+', '_', str(name))
        name = re.sub(r'\s+', '_', name)
        name = re.sub(r'[^a-zA-Z0-9_]', '', name)
        if name and name[0].isdigit():
            name = 'f_' + name
        if not name:
            name = 'feature'
        return name
    
    df.columns = [clean_name(col) for col in df.columns]
    return df

def calculate_nrmse(y_true, y_pred, method='mean'):
    """Calculate normalized RMSE."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    if method == 'mean':
        return rmse / np.mean(y_true)
    elif method == 'range':
        return rmse / (np.max(y_true) - np.min(y_true))
    else:
        return rmse / np.std(y_true)

def bootstrap_metrics(y_true, y_pred, n_bootstrap=1000, random_state=42):
    """Calculate metrics with bootstrap confidence intervals."""
    np.random.seed(random_state)
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    
    metrics = {
        'R2': [],
        'RMSE': [],
        'MAE': [],
        'MAPE': []
    }
    
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        y_true_boot = y_true[idx]
        y_pred_boot = y_pred[idx]
        
        metrics['R2'].append(r2_score(y_true_boot, y_pred_boot))
        metrics['RMSE'].append(np.sqrt(mean_squared_error(y_true_boot, y_pred_boot)))
        metrics['MAE'].append(mean_absolute_error(y_true_boot, y_pred_boot))
        metrics['MAPE'].append(mean_absolute_percentage_error(y_true_boot, y_pred_boot))
    
    results = {}
    for metric_name, values in metrics.items():
        values = np.array(values)
        results[metric_name] = {
            'mean': np.mean(values),
            'std': np.std(values),
            'lower_95': np.percentile(values, 2.5),
            'upper_95': np.percentile(values, 97.5)
        }
    
    return results

def calculate_residual_metrics(y_true, y_pred):
    """Calculate comprehensive residual statistics."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    residuals = y_true - y_pred
    standardized_residuals = residuals / np.std(residuals) if np.std(residuals) > 0 else residuals
    
    return {
        'mean': np.mean(residuals),
        'std': np.std(residuals),
        'min': np.min(residuals),
        'max': np.max(residuals),
        'skew': stats.skew(residuals),
        'kurtosis': stats.kurtosis(residuals),
        'shapiro_wilk_p': stats.shapiro(residuals)[1] if len(residuals) >= 3 else np.nan,
    }

def calculate_ci_from_scores(scores, confidence=0.95):
    """Calculate confidence intervals from a list of scores."""
    scores = np.array(scores)
    n = len(scores)
    mean = np.mean(scores)
    std = np.std(scores, ddof=1)
    se = sem(scores)
    
    t_val = t.ppf((1 + confidence) / 2, n - 1)
    ci_lower = mean - t_val * se
    ci_upper = mean + t_val * se
    
    return {
        'mean': mean,
        'std': std,
        'se': se,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'n': n
    }

# ============================================================================
# PREPROCESSOR - SIMPLIFIED FOR TREE-BASED MODELS
# ============================================================================
class TreePreprocessor(BaseEstimator, TransformerMixin):
    """
    Simple preprocessor for tree-based models.
    - Handles missing values (median imputation)
    - NO feature selection (trees handle this natively)
    - NO scaling (not needed for trees)
    """
    def __init__(self, handle_missing: str = 'median'):
        self.handle_missing = handle_missing
        self.imputer = None
        self.feature_names_in_ = None
        
    def fit(self, X: pd.DataFrame, y: pd.Series = None):
        X_copy = X.copy()
        self.feature_names_in_ = X_copy.columns.tolist()
        
        # Handle missing values
        if self.handle_missing == 'median':
            self.imputer = SimpleImputer(strategy='median')
        elif self.handle_missing == 'mean':
            self.imputer = SimpleImputer(strategy='mean')
        elif self.handle_missing == 'constant':
            self.imputer = SimpleImputer(strategy='constant', fill_value=0)
        else:
            self.imputer = SimpleImputer(strategy='median')
        
        self.imputer.fit(X_copy)
        return self
    
    def transform(self, X: pd.DataFrame):
        X_copy = X.copy()
        
        if self.imputer is not None:
            X_imputed = self.imputer.transform(X_copy)
            X_imputed = pd.DataFrame(X_imputed, columns=X_copy.columns, index=X_copy.index)
        else:
            X_imputed = X_copy
        
        return X_imputed

# ============================================================================
# SCIENTIFIC ADABOOST - 80/20 SPLIT WITH REPEATED CV
# ============================================================================
class ScientificAdaBoost:
    """
    AdaBoost with rigorous 80/20 split and repeated cross-validation.
    
    WORKFLOW:
    1. Single 80/20 train/test split (stored in model)
    2. RepeatedKFold (5×3 = 15 splits) for hyperparameter tuning
    3. Retrain best model on full 80%
    4. Evaluate on held-out 20%
    5. Permutation importance for feature importance
    6. Bootstrap CIs for all metrics
    """
    
    def __init__(
        self,
        config: Dict[str, Any] = None,
        output_dir: str = None
    ):
        self.config = config or self._default_config()
        self.output_dir = output_dir or r"D:\2026 Work\My Papers\Chloride Ingress\Models\AdaBoost"
        
        self.pipeline = None
        self.best_params = {}
        self.metrics_train = {}
        self.metrics_test = {}
        self.metrics_test_bootstrap = {}
        self.metrics_cv = {}
        self.metrics_cv_bootstrap = {}
        self.cv_predictions = None
        self.grid_search_results = None
        self.original_feature_names = None
        self.reproducibility_info = None
        self.residual_analysis = {}
        self.permutation_importance = None
        
        # Store data splits
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        
        # Split sizes
        self.n_train_samples = 0
        self.n_test_samples = 0
        self.n_features = 0
        
        self._setup_directories()
        
    def _setup_directories(self):
        for subdir in ['Plots', 'Tables', 'Models', 'Reports']:
            os.makedirs(os.path.join(self.output_dir, subdir), exist_ok=True)
    
    def _default_config(self) -> Dict[str, Any]:
        return {
            'test_size': 0.20,
            'random_state': 42,
            'cv_folds': 5,
            'cv_repeats': 3,  # 5×3 = 15 splits
            'n_jobs': -1,
            'cv_scoring': 'neg_mean_squared_error',
            'n_bootstrap': 1000,
            'n_permutation_repeats': 30
        }
    
    def _create_pipeline(self, preprocessor=None):
        """Create the AdaBoost pipeline."""
        if preprocessor is None:
            preprocessor = TreePreprocessor(handle_missing='median')
        
        # Base estimator with default params (will be tuned)
        base_estimator = DecisionTreeRegressor(
            max_depth=4,
            min_samples_split=5,
            min_samples_leaf=3,
            random_state=self.config['random_state']
        )
        
        # AdaBoost model
        model = AdaBoostRegressor(
            estimator=base_estimator,
            n_estimators=100,
            learning_rate=0.1,
            loss='square',
            random_state=self.config['random_state']
        )
        
        self.pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model)
        ])
        
        return self.pipeline
    
    def _get_practical_grid(self) -> Dict:
        """
        PRACTICAL GRID SEARCH:
        Focused parameter ranges that cover the most important hyperparameters.
        
        Total combinations: 3 × 3 × 3 × 2 × 2 × 2 = 216
        With 5×3 repeated CV: 216 × 15 = 3,240 model fits
        """
        return {
            'model__n_estimators': [50, 100, 200],
            'model__learning_rate': [0.05, 0.1, 0.2],
            'model__estimator__max_depth': [2, 3, 4],
            'model__estimator__min_samples_split': [2, 5],
            'model__estimator__min_samples_leaf': [1, 2],
            'model__loss': ['linear', 'square']
        }
    
    def _create_repeated_cv(self):
        """Create RepeatedKFold with shuffling for reproducibility."""
        return RepeatedKFold(
            n_splits=self.config['cv_folds'],
            n_repeats=self.config['cv_repeats'],
            random_state=self.config['random_state']
        )
    
    def train(self, X: pd.DataFrame, y: pd.Series):
        """
        Train with 80/20 split and repeated cross-validation.
        """
        logger.info("=" * 80)
        logger.info("SCIENTIFIC ADABOOST - 80/20 SPLIT WITH REPEATED CV")
        logger.info("=" * 80)
        
        self.original_feature_names = X.columns.tolist()
        self.n_features = len(X.columns)
        self.reproducibility_info = generate_reproducibility_report()
        
        # ====================================================================
        # STEP 1: SINGLE 80/20 TRAIN/TEST SPLIT (STORED IN MODEL)
        # ====================================================================
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y,
            test_size=self.config['test_size'],
            random_state=self.config['random_state'],
            shuffle=True
        )
        
        self.n_train_samples = len(self.X_train)
        self.n_test_samples = len(self.X_test)
        
        logger.info(f"Data Split (80/20):")
        logger.info(f"  Training: {self.n_train_samples} samples ({self.n_train_samples/(self.n_train_samples+self.n_test_samples):.1%})")
        logger.info(f"  Testing : {self.n_test_samples} samples ({self.n_test_samples/(self.n_train_samples+self.n_test_samples):.1%})")
        
        # ====================================================================
        # STEP 2: REPEATED CROSS-VALIDATION (5×3 = 15 splits)
        # ====================================================================
        param_grid = self._get_practical_grid()
        total_combinations = np.prod([len(v) for v in param_grid.values()])
        total_fits = total_combinations * self.config['cv_folds'] * self.config['cv_repeats']
        
        logger.info(f"\nGridSearchCV with RepeatedKFold (5×3 = 15 splits):")
        logger.info(f"  Parameter combinations: {total_combinations}")
        logger.info(f"  Total fits: {total_fits}")
        logger.info(f"  Estimated time: ~{total_fits / 50:.1f} seconds (with n_jobs=-1)")
        
        preprocessor = TreePreprocessor(handle_missing='median')
        pipeline = self._create_pipeline(preprocessor)
        
        # Use RepeatedKFold for reproducible CV
        cv = self._create_repeated_cv()
        
        grid_search = GridSearchCV(
            pipeline,
            param_grid,
            cv=cv,
            scoring=self.config['cv_scoring'],
            n_jobs=self.config['n_jobs'],
            verbose=0,
            return_train_score=True
        )
        
        start_time = time.time()
        grid_search.fit(self.X_train, self.y_train)
        elapsed_time = time.time() - start_time
        
        self.grid_search_results = grid_search
        self.best_params = grid_search.best_params_
        
        logger.info(f"\nGridSearchCV completed in {elapsed_time:.1f} seconds")
        logger.info(f"Best parameters: {self.best_params}")
        logger.info(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")
        
        # ====================================================================
        # STEP 3: CROSS-VALIDATION METRICS (FOR PUBLICATION)
        # ====================================================================
        logger.info("\nCalculating cross-validation metrics...")
        
        # Get all CV scores from grid search results
        cv_results = pd.DataFrame(grid_search.cv_results_)
        
        # Get the best model's individual fold scores
        best_index = grid_search.best_index_
        
        # Properly extract split scores and convert to numeric
        split_cols = [col for col in cv_results.columns if 'split' in col and 'test_score' in col]
        if split_cols:
            best_cv_scores = cv_results.loc[best_index, split_cols].values
            best_cv_scores = np.array(best_cv_scores, dtype=np.float64)
            best_cv_scores = best_cv_scores[~np.isnan(best_cv_scores)]
            best_cv_rmse = np.sqrt(-best_cv_scores)
        else:
            best_cv_rmse = np.array([np.sqrt(-grid_search.best_score_)])
        
        # ====================================================================
        # FIX: Manual cross-validation prediction for RepeatedKFold
        # ====================================================================
        best_pipeline = grid_search.best_estimator_
        cv = self._create_repeated_cv()
        
        # Manually perform cross-validation predictions
        cv_predictions = np.zeros(len(self.X_train))
        cv_indices = np.zeros(len(self.X_train), dtype=int)
        
        fold_idx = 0
        for train_idx, val_idx in cv.split(self.X_train):
            # Train on train_idx
            X_train_fold = self.X_train.iloc[train_idx]
            y_train_fold = self.y_train.iloc[train_idx]
            
            # Create a fresh pipeline for this fold
            fold_pipeline = self._create_pipeline(preprocessor)
            fold_pipeline.fit(X_train_fold, y_train_fold)
            
            # Predict on val_idx
            X_val_fold = self.X_train.iloc[val_idx]
            y_pred_fold = fold_pipeline.predict(X_val_fold)
            
            # Store predictions
            cv_predictions[val_idx] = y_pred_fold
            cv_indices[val_idx] = fold_idx
            fold_idx += 1
        
        self.cv_predictions = cv_predictions
        
        # Calculate metrics from CV predictions
        self.metrics_cv = {
            'R2_mean': r2_score(self.y_train, cv_predictions),
            'RMSE_mean': np.sqrt(mean_squared_error(self.y_train, cv_predictions)),
            'MAE_mean': mean_absolute_error(self.y_train, cv_predictions)
        }
        
        # Calculate per-fold metrics
        fold_metrics = []
        for fold_i in range(fold_idx):
            mask = cv_indices == fold_i
            if np.sum(mask) > 0:
                y_val_fold = self.y_train[mask]
                y_pred_fold = cv_predictions[mask]
                
                fold_metrics.append({
                    'fold': fold_i,
                    'R2': r2_score(y_val_fold, y_pred_fold),
                    'RMSE': np.sqrt(mean_squared_error(y_val_fold, y_pred_fold)),
                    'MAE': mean_absolute_error(y_val_fold, y_pred_fold)
                })
        
        self.metrics_cv['fold_metrics'] = fold_metrics
        
        # Calculate CV statistics
        cv_r2_scores = [m['R2'] for m in fold_metrics]
        cv_rmse_scores = [m['RMSE'] for m in fold_metrics]
        
        self.metrics_cv['R2_stats'] = calculate_ci_from_scores(cv_r2_scores)
        self.metrics_cv['RMSE_stats'] = calculate_ci_from_scores(cv_rmse_scores)
        
        logger.info(f"\nCross-Validation Metrics (5×3 = 15 folds):")
        logger.info(f"  R²  : {self.metrics_cv['R2_stats']['mean']:.4f} ± {self.metrics_cv['R2_stats']['std']:.4f}")
        logger.info(f"  R² 95% CI: [{self.metrics_cv['R2_stats']['ci_lower']:.4f}, {self.metrics_cv['R2_stats']['ci_upper']:.4f}]")
        logger.info(f"  RMSE: {self.metrics_cv['RMSE_stats']['mean']:.4f} ± {self.metrics_cv['RMSE_stats']['std']:.4f}")
        logger.info(f"  RMSE 95% CI: [{self.metrics_cv['RMSE_stats']['ci_lower']:.4f}, {self.metrics_cv['RMSE_stats']['ci_upper']:.4f}]")
        
        # ====================================================================
        # STEP 4: RETRAIN ON FULL TRAINING SET
        # ====================================================================
        logger.info("\nRetraining best model on full training set...")
        self.pipeline = grid_search.best_estimator_
        
        # ====================================================================
        # STEP 5: EVALUATE ON TRAINING SET
        # ====================================================================
        y_train_pred = self.pipeline.predict(self.X_train)
        
        self.metrics_train = {
            'R2': r2_score(self.y_train, y_train_pred),
            'RMSE': np.sqrt(mean_squared_error(self.y_train, y_train_pred)),
            'MAE': mean_absolute_error(self.y_train, y_train_pred)
        }
        
        logger.info(f"\nTraining Set Metrics:")
        logger.info(f"  R²: {self.metrics_train['R2']:.4f}")
        logger.info(f"  RMSE: {self.metrics_train['RMSE']:.4f}")
        
        # ====================================================================
        # STEP 6: EVALUATE ON TEST SET
        # ====================================================================
        y_test_pred = self.pipeline.predict(self.X_test)
        
        self.metrics_test = {
            'R2': r2_score(self.y_test, y_test_pred),
            'RMSE': np.sqrt(mean_squared_error(self.y_test, y_test_pred)),
            'MAE': mean_absolute_error(self.y_test, y_test_pred),
            'MAPE': mean_absolute_percentage_error(self.y_test, y_test_pred),
            'NRMSE_mean': calculate_nrmse(self.y_test, y_test_pred, method='mean'),
            'NRMSE_range': calculate_nrmse(self.y_test, y_test_pred, method='range'),
            'Pearson': pearsonr(self.y_test, y_test_pred)[0],
            'Spearman': spearmanr(self.y_test, y_test_pred)[0]
        }
        
        logger.info(f"\nTest Set Metrics:")
        for metric, value in self.metrics_test.items():
            logger.info(f"  {metric:15s}: {value:.4f}")
        
        # ====================================================================
        # STEP 7: BOOTSTRAP CONFIDENCE INTERVALS
        # ====================================================================
        logger.info("\nCalculating bootstrap confidence intervals...")
        
        # Test set bootstrap
        self.metrics_test_bootstrap = bootstrap_metrics(
            self.y_test, y_test_pred,
            n_bootstrap=self.config['n_bootstrap'],
            random_state=self.config['random_state']
        )
        
        logger.info(f"\nTest Set Bootstrap 95% Confidence Intervals:")
        for metric, values in self.metrics_test_bootstrap.items():
            logger.info(f"  {metric:15s}: {values['mean']:.4f} [{values['lower_95']:.4f}, {values['upper_95']:.4f}]")
        
        # CV bootstrap (using CV predictions)
        self.metrics_cv_bootstrap = bootstrap_metrics(
            self.y_train, self.cv_predictions,
            n_bootstrap=self.config['n_bootstrap'],
            random_state=self.config['random_state'] + 1
        )
        
        logger.info(f"\nCV Bootstrap 95% Confidence Intervals:")
        for metric, values in self.metrics_cv_bootstrap.items():
            logger.info(f"  {metric:15s}: {values['mean']:.4f} [{values['lower_95']:.4f}, {values['upper_95']:.4f}]")
        
        # ====================================================================
        # STEP 8: RESIDUAL ANALYSIS
        # ====================================================================
        self.residual_analysis = {
            'test': calculate_residual_metrics(self.y_test, y_test_pred),
            'training': calculate_residual_metrics(self.y_train, y_train_pred),
            'cv': calculate_residual_metrics(self.y_train, self.cv_predictions)
        }
        
        # ====================================================================
        # STEP 9: PERMUTATION IMPORTANCE (FOR PUBLICATION)
        # ====================================================================
        logger.info("\nCalculating permutation importance...")
        
        try:
            perm_result = permutation_importance(
                self.pipeline,
                self.X_test,
                self.y_test,
                n_repeats=self.config['n_permutation_repeats'],
                random_state=self.config['random_state'],
                scoring='r2',
                n_jobs=self.config['n_jobs']
            )
            
            # Get feature names
            try:
                feature_names = self.pipeline.named_steps['preprocessor'].feature_names_in_
            except:
                feature_names = self.original_feature_names
            
            self.permutation_importance = pd.DataFrame({
                'Feature': feature_names[:len(perm_result.importances_mean)],
                'Importance_mean': perm_result.importances_mean,
                'Importance_std': perm_result.importances_std
            }).sort_values('Importance_mean', ascending=False)
            
            logger.info(f"  Top 5 features by permutation importance:")
            for i, row in self.permutation_importance.head(5).iterrows():
                logger.info(f"    {row['Feature']:30s}: {row['Importance_mean']:.4f} ± {row['Importance_std']:.4f}")
                
        except Exception as e:
            logger.warning(f"Permutation importance failed: {e}")
            self.permutation_importance = None
        
        # ====================================================================
        # STEP 10: SUMMARY
        # ====================================================================
        logger.info("\n" + "=" * 80)
        logger.info("TRAINING COMPLETE - SUMMARY")
        logger.info("=" * 80)
        logger.info(f"  Training samples: {self.n_train_samples}")
        logger.info(f"  Test samples: {self.n_test_samples}")
        logger.info(f"  Features: {self.n_features}")
        logger.info(f"  CV R²: {self.metrics_cv['R2_stats']['mean']:.4f} ± {self.metrics_cv['R2_stats']['std']:.4f}")
        logger.info(f"  Test R²: {self.metrics_test['R2']:.4f}")
        logger.info(f"  Test RMSE: {self.metrics_test['RMSE']:.4f}")
        logger.info(f"  Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")
        logger.info(f"  Grid combinations: {total_combinations}")
        logger.info(f"  CV splits: {self.config['cv_folds']}×{self.config['cv_repeats']} = {self.config['cv_folds'] * self.config['cv_repeats']}")
        logger.info(f"  Total CV fits: {total_fits}")
        logger.info("=" * 80)
        
        return self
    
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Make predictions."""
        return self.pipeline.predict(X)
    
    def get_feature_importance(self, method='permutation') -> pd.DataFrame:
        """
        Get feature importance.
        
        Parameters:
        -----------
        method : str
            'permutation' - Permutation importance (preferred for publication)
            'impurity' - Built-in impurity-based importance
        """
        if method == 'permutation' and self.permutation_importance is not None:
            return self.permutation_importance
        elif method == 'impurity':
            try:
                model = self.pipeline.named_steps['model']
                importance = model.feature_importances_
                
                try:
                    feature_names = self.pipeline.named_steps['preprocessor'].feature_names_in_
                except:
                    feature_names = self.original_feature_names
                
                if len(importance) != len(feature_names):
                    feature_names = [f'Feature_{i}' for i in range(len(importance))]
                
                return pd.DataFrame({
                    'Feature': feature_names[:len(importance)],
                    'Importance': importance
                }).sort_values('Importance', ascending=False)
            except Exception as e:
                logger.warning(f"Could not extract impurity importance: {e}")
                return pd.DataFrame()
        else:
            return pd.DataFrame()
    
    def save_model(self, filepath: str = None):
        """Save the model."""
        if filepath is None:
            filepath = os.path.join(self.output_dir, 'Models', 'AdaBoost_80_20.joblib')
        
        model_to_save = {
            'pipeline': self.pipeline,
            'best_params': self.best_params,
            'metrics_train': self.metrics_train,
            'metrics_test': self.metrics_test,
            'metrics_test_bootstrap': self.metrics_test_bootstrap,
            'metrics_cv': self.metrics_cv,
            'metrics_cv_bootstrap': self.metrics_cv_bootstrap,
            'permutation_importance': self.permutation_importance,
            'config': self.config,
            'reproducibility_info': self.reproducibility_info,
            'n_train_samples': self.n_train_samples,
            'n_test_samples': self.n_test_samples,
            'n_features': self.n_features,
            'original_feature_names': self.original_feature_names
        }
        
        joblib.dump(model_to_save, filepath)
        logger.info(f"Model saved to: {filepath}")
        
        return filepath

# ============================================================================
# VISUALIZER
# ============================================================================
class ScientificVisualizer:
    def __init__(self, output_dir: str, dpi: int = 300):
        self.output_dir = output_dir
        self.plots_dir = os.path.join(output_dir, "Plots")
        self.dpi = dpi
        os.makedirs(self.plots_dir, exist_ok=True)
        
        plt.style.use('seaborn-v0_8-whitegrid')
        self.colors = {
            'blue': '#2E86AB',
            'red': '#D62828',
            'green': '#2E9F6E',
            'purple': '#7B2D8B',
            'orange': '#F77F00',
            'grey': '#6C6C6C'
        }
    
    def generate_all_plots(self, model: ScientificAdaBoost):
        """Generate all publication-quality plots."""
        logger.info("Generating publication-quality plots...")
        
        y_train_pred = model.predict(model.X_train)
        y_test_pred = model.predict(model.X_test)
        
        # Figure 1: Scatter plot (Actual vs Predicted) - Test Set
        self._plot_scatter(model.y_test, y_test_pred, 'Test Set: Actual vs Predicted',
                          os.path.join(self.plots_dir, 'Fig1_Scatter_Test.png'))
        
        # Figure 2: Scatter plot - CV Predictions
        if hasattr(model, 'cv_predictions') and model.cv_predictions is not None:
            self._plot_scatter(model.y_train, model.cv_predictions, 
                              'Cross-Validation: Actual vs Predicted',
                              os.path.join(self.plots_dir, 'Fig2_Scatter_CV.png'))
        
        # Figure 3: Residual analysis - Test Set
        self._plot_residuals(model.y_test, y_test_pred,
                            'Test Set Residual Analysis',
                            os.path.join(self.plots_dir, 'Fig3_Residuals_Test.png'))
        
        # Figure 4: Residual analysis - CV
        if hasattr(model, 'cv_predictions') and model.cv_predictions is not None:
            self._plot_residuals(model.y_train, model.cv_predictions,
                                'Cross-Validation Residual Analysis',
                                os.path.join(self.plots_dir, 'Fig4_Residuals_CV.png'))
        
        # Figure 5: Q-Q plot - Test Set
        self._plot_qq(model.y_test, y_test_pred,
                     os.path.join(self.plots_dir, 'Fig5_QQ_Test.png'))
        
        # Figure 6: Q-Q plot - CV
        if hasattr(model, 'cv_predictions') and model.cv_predictions is not None:
            self._plot_qq(model.y_train, model.cv_predictions,
                         os.path.join(self.plots_dir, 'Fig6_QQ_CV.png'))
        
        # Figure 7: Feature importance (Permutation)
        importance_df = model.get_feature_importance(method='permutation')
        if not importance_df.empty:
            self._plot_feature_importance(importance_df.head(15),
                                         os.path.join(self.plots_dir, 'Fig7_Permutation_Importance.png'))
        
        # Figure 8: CV Performance
        if model.grid_search_results is not None:
            self._plot_cv_performance(model,
                                     os.path.join(self.plots_dir, 'Fig8_CV_Performance.png'))
        
        # Figure 9: CV Fold Performance
        if 'fold_metrics' in model.metrics_cv:
            self._plot_cv_folds(model.metrics_cv['fold_metrics'],
                               os.path.join(self.plots_dir, 'Fig9_CV_Folds.png'))
        
        logger.info(f"Plots saved to: {self.plots_dir}")
    
    def _plot_scatter(self, y_true, y_pred, title, filename):
        fig, ax = plt.subplots(figsize=(8, 8))
        
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        
        ax.scatter(y_true, y_pred, c=self.colors['blue'], s=40, alpha=0.6,
                  edgecolors='black', linewidth=0.5)
        
        min_val = min(y_true.min(), y_pred.min()) - 0.1
        max_val = max(y_true.max(), y_pred.max()) + 0.1
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5)
        
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        
        textstr = f'$R^2$ = {r2:.4f}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}'
        props = dict(boxstyle='round', facecolor='white', edgecolor='black', linewidth=1.5, alpha=0.9)
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
                verticalalignment='top', bbox=props)
        
        ax.set_xlabel('Actual Total Chloride (% of binder mass)', fontsize=12)
        ax.set_ylabel('Predicted Total Chloride (% of binder mass)', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_residuals(self, y_true, y_pred, title, filename):
        fig, ax = plt.subplots(figsize=(10, 6))
        
        residuals = y_true - y_pred
        fitted = y_pred
        
        ax.scatter(fitted, residuals, c=self.colors['blue'], s=40, alpha=0.6,
                  edgecolors='black', linewidth=0.5)
        
        ax.axhline(y=0, color=self.colors['red'], linestyle='--', linewidth=1.5)
        
        mean_res = np.mean(residuals)
        std_res = np.std(residuals)
        
        textstr = f'Mean: {mean_res:.4f}\nStd: {std_res:.4f}'
        props = dict(boxstyle='round', facecolor='white', edgecolor='black', linewidth=1.5, alpha=0.9)
        ax.text(0.95, 0.95, textstr, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', ha='right', bbox=props)
        
        ax.set_xlabel('Predicted Total Chloride (% of binder mass)', fontsize=12)
        ax.set_ylabel('Residual (Actual - Predicted)', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_qq(self, y_true, y_pred, filename):
        residuals = y_true - y_pred
        standardized_residuals = (residuals - np.mean(residuals)) / np.std(residuals)
        
        fig, ax = plt.subplots(figsize=(8, 8))
        
        stats.probplot(standardized_residuals, dist="norm", plot=ax)
        
        ax.set_title('Q-Q Plot of Standardized Residuals', fontsize=14, fontweight='bold')
        ax.set_xlabel('Theoretical Quantiles', fontsize=12)
        ax.set_ylabel('Sample Quantiles', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        if len(residuals) >= 3:
            _, p_value = stats.shapiro(residuals)
            ax.text(0.05, 0.95, f'Shapiro-Wilk p = {p_value:.4f}', 
                   transform=ax.transAxes, fontsize=10,
                   verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_feature_importance(self, importance_df, filename):
        fig, ax = plt.subplots(figsize=(10, 8))
        
        importance_df = importance_df.head(15)
        
        colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(importance_df)))[::-1]
        
        # Check if we have std (permutation importance)
        if 'Importance_std' in importance_df.columns:
            ax.barh(importance_df['Feature'], importance_df['Importance_mean'], 
                   xerr=importance_df['Importance_std'],
                   color=colors, edgecolor='black', linewidth=0.5,
                   capsize=3)
            xlabel = 'Permutation Importance (R² decrease)'
        else:
            ax.barh(importance_df['Feature'], importance_df['Importance'], 
                   color=colors, edgecolor='black', linewidth=0.5)
            xlabel = 'Feature Importance'
        
        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel('Feature', fontsize=12)
        ax.set_title('Top 15 Feature Importances (Permutation)', fontsize=14, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_cv_performance(self, model, filename):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        results = model.grid_search_results
        cv_results = pd.DataFrame(results.cv_results_)
        
        mean_scores = -cv_results['mean_test_score']
        std_scores = cv_results['std_test_score']
        
        # Plot 1: Performance vs n_estimators
        n_estimators_values = [p['model__n_estimators'] for p in cv_results['params']]
        
        grouped = pd.DataFrame({
            'n_estimators': n_estimators_values,
            'mean_rmse': mean_scores,
        }).groupby('n_estimators').agg({
            'mean_rmse': ['mean', 'std']
        }).reset_index()
        
        grouped.columns = ['n_estimators', 'mean_rmse', 'std_rmse']
        
        ax1.errorbar(grouped['n_estimators'], grouped['mean_rmse'], 
                    yerr=grouped['std_rmse'], fmt='o-', 
                    color=self.colors['blue'], capsize=5, capthick=2,
                    markersize=8, linewidth=2)
        
        ax1.set_xlabel('Number of Estimators', fontsize=12)
        ax1.set_ylabel('RMSE (CV)', fontsize=12)
        ax1.set_title('CV Performance vs n_estimators', fontsize=12, fontweight='bold')
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Performance vs learning_rate
        learning_rate_values = [p['model__learning_rate'] for p in cv_results['params']]
        
        grouped2 = pd.DataFrame({
            'learning_rate': learning_rate_values,
            'mean_rmse': mean_scores,
        }).groupby('learning_rate').agg({
            'mean_rmse': ['mean', 'std']
        }).reset_index()
        
        grouped2.columns = ['learning_rate', 'mean_rmse', 'std_rmse']
        
        ax2.errorbar(grouped2['learning_rate'], grouped2['mean_rmse'],
                    yerr=grouped2['std_rmse'], fmt='o-',
                    color=self.colors['red'], capsize=5, capthick=2,
                    markersize=8, linewidth=2)
        
        ax2.set_xlabel('Learning Rate', fontsize=12)
        ax2.set_ylabel('RMSE (CV)', fontsize=12)
        ax2.set_title('CV Performance vs Learning Rate', fontsize=12, fontweight='bold')
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(False)
        ax2.grid(True, alpha=0.3)
        
        plt.suptitle('GridSearchCV Hyperparameter Sensitivity (Repeated CV)', 
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _plot_cv_folds(self, fold_metrics, filename):
        fig, ax = plt.subplots(figsize=(10, 6))
        
        fold_numbers = [m['fold'] + 1 for m in fold_metrics]
        r2_scores = [m['R2'] for m in fold_metrics]
        rmse_scores = [m['RMSE'] for m in fold_metrics]
        
        x = np.arange(len(fold_numbers))
        width = 0.35
        
        ax.bar(x - width/2, r2_scores, width, label='R²', color=self.colors['blue'], alpha=0.7)
        
        ax2 = ax.twinx()
        ax2.bar(x + width/2, rmse_scores, width, label='RMSE', color=self.colors['red'], alpha=0.7)
        
        ax.axhline(y=np.mean(r2_scores), color='blue', linestyle='--', linewidth=1.5, alpha=0.7)
        ax2.axhline(y=np.mean(rmse_scores), color='red', linestyle='--', linewidth=1.5, alpha=0.7)
        
        ax.set_xlabel('CV Fold', fontsize=12)
        ax.set_ylabel('R²', fontsize=12, color='blue')
        ax2.set_ylabel('RMSE', fontsize=12, color='red')
        ax.set_title('Cross-Validation Performance by Fold (5×3 = 15 folds)', 
                    fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(fold_numbers)
        
        # Add legend
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(True)
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        plt.savefig(filename, dpi=self.dpi, bbox_inches='tight', facecolor='white')
        plt.close()

# ============================================================================
# MAIN TRAINER
# ============================================================================
class ScientificModelTrainer:
    def __init__(self, config: Dict[str, Any] = None):
        self.config = config or {}
        self.results_dir = r"D:\2026 Work\My Papers\Chloride Ingress\Models\AdaBoost"
        
        for subdir in ['Plots', 'Tables', 'Models', 'Reports']:
            os.makedirs(os.path.join(self.results_dir, subdir), exist_ok=True)
        
        self.model = None
        self.visualizer = ScientificVisualizer(self.results_dir)
    
    def load_data(self, filepath: str) -> Tuple[pd.DataFrame, pd.Series, str]:
        logger.info(f"Loading data from: {filepath}")
        
        df = pd.read_csv(filepath, encoding='ISO-8859-1')
        df.columns = df.columns.str.strip()
        
        logger.info(f"Initial data shape: {df.shape}")
        
        target_col = 'Total chloride, % of binder mass'
        
        if target_col not in df.columns:
            available_cols = df.columns.tolist()
            raise ValueError(
                f"Target column '{target_col}' not found.\n"
                f"Available columns: {available_cols}\n"
                f"Please check the column name."
            )
        
        # Remove exposure zone columns (data leakage)
        features_to_drop = ['Exposure_zones', 'Exposure Zones', 'ExposureZone', 'exposure_zones']
        for feature in features_to_drop:
            if feature in df.columns:
                df = df.drop(columns=[feature])
                logger.warning(f"REMOVED '{feature}' to prevent data leakage")
        
        X = df.drop(columns=[target_col])
        y = df[target_col]
        
        # Handle NaN in target
        nan_count_y = y.isna().sum()
        if nan_count_y > 0:
            logger.warning(f"Found {nan_count_y} NaN values in target")
            valid_idx = ~y.isna()
            X = X[valid_idx]
            y = y[valid_idx]
            logger.info(f"Removed {nan_count_y} rows. New shape: {X.shape}")
        
        X = sanitize_feature_names(X)
        
        logger.info(f"Loaded {len(X)} samples with {len(X.columns)} features")
        logger.info(f"Target stats - Min: {y.min():.4f}, Max: {y.max():.4f}, Mean: {y.mean():.4f}")
        
        return X, y, target_col
    
    def run_pipeline(self, data_path: str) -> ScientificAdaBoost:
        logger.info("=" * 80)
        logger.info("SCIENTIFIC ADABOOST - TOTAL CHLORIDE PREDICTION")
        logger.info("VERSION 4.0 - REPEATED CV + PERMUTATION IMPORTANCE")
        logger.info("=" * 80)
        logger.info("\nHYPERPARAMETER GRID:")
        logger.info("  n_estimators    : [50, 100, 200]")
        logger.info("  learning_rate   : [0.05, 0.1, 0.2]")
        logger.info("  max_depth       : [2, 3, 4]")
        logger.info("  min_samples_split: [2, 5]")
        logger.info("  min_samples_leaf: [1, 2]")
        logger.info("  loss            : ['linear', 'square']")
        logger.info(f"  Total combinations: 216")
        logger.info(f"  CV: RepeatedKFold (5×3 = 15 splits)")
        logger.info(f"  Total CV fits: 3,240")
        logger.info("\nFEATURE IMPORTANCE:")
        logger.info("  Method: Permutation Importance (preferred for publication)")
        logger.info("  Repeats: 30")
        logger.info("=" * 80 + "\n")
        
        X, y, target_col = self.load_data(data_path)
        
        # Model configuration
        model_config = {
            'test_size': 0.20,
            'random_state': 42,
            'cv_folds': 5,
            'cv_repeats': 3,
            'n_jobs': -1,
            'cv_scoring': 'neg_mean_squared_error',
            'n_bootstrap': 1000,
            'n_permutation_repeats': 30
        }
        
        self.model = ScientificAdaBoost(
            model_config,
            output_dir=self.results_dir
        )
        
        # Train model (split is done inside and stored)
        self.model.train(X, y)
        
        # Generate plots (using stored splits)
        self.visualizer.generate_all_plots(self.model)
        
        # Save model
        self.model.save_model()
        
        # Save results
        self._save_results(self.model)
        
        return self.model
    
    def _save_results(self, model: ScientificAdaBoost):
        """Save all results as CSV and JSON."""
        
        # Test metrics
        metrics_df = pd.DataFrame({
            'Metric': list(model.metrics_test.keys()),
            'Value': [model.metrics_test[m] for m in model.metrics_test.keys()]
        })
        metrics_df.to_csv(os.path.join(self.results_dir, 'Tables', 'test_metrics.csv'), index=False)
        
        # CV metrics
        cv_metrics = {
            'R2_mean': model.metrics_cv['R2_stats']['mean'],
            'R2_std': model.metrics_cv['R2_stats']['std'],
            'R2_ci_lower': model.metrics_cv['R2_stats']['ci_lower'],
            'R2_ci_upper': model.metrics_cv['R2_stats']['ci_upper'],
            'RMSE_mean': model.metrics_cv['RMSE_stats']['mean'],
            'RMSE_std': model.metrics_cv['RMSE_stats']['std'],
            'RMSE_ci_lower': model.metrics_cv['RMSE_stats']['ci_lower'],
            'RMSE_ci_upper': model.metrics_cv['RMSE_stats']['ci_upper'],
            'n_folds': len(model.metrics_cv['fold_metrics'])
        }
        cv_df = pd.DataFrame([cv_metrics])
        cv_df.to_csv(os.path.join(self.results_dir, 'Tables', 'cv_metrics.csv'), index=False)
        
        # CV fold metrics
        if 'fold_metrics' in model.metrics_cv:
            fold_df = pd.DataFrame(model.metrics_cv['fold_metrics'])
            fold_df.to_csv(os.path.join(self.results_dir, 'Tables', 'cv_fold_metrics.csv'), index=False)
        
        # Test bootstrap CIs
        if model.metrics_test_bootstrap:
            bootstrap_rows = []
            for metric, values in model.metrics_test_bootstrap.items():
                bootstrap_rows.append({
                    'Metric': metric,
                    'Mean': values['mean'],
                    'Std': values['std'],
                    '2.5% CI': values['lower_95'],
                    '97.5% CI': values['upper_95']
                })
            bootstrap_df = pd.DataFrame(bootstrap_rows)
            bootstrap_df.to_csv(os.path.join(self.results_dir, 'Tables', 'bootstrap_ci_test.csv'), index=False)
        
        # CV bootstrap CIs
        if hasattr(model, 'metrics_cv_bootstrap'):
            bootstrap_rows = []
            for metric, values in model.metrics_cv_bootstrap.items():
                bootstrap_rows.append({
                    'Metric': metric,
                    'Mean': values['mean'],
                    'Std': values['std'],
                    '2.5% CI': values['lower_95'],
                    '97.5% CI': values['upper_95']
                })
            bootstrap_df = pd.DataFrame(bootstrap_rows)
            bootstrap_df.to_csv(os.path.join(self.results_dir, 'Tables', 'bootstrap_ci_cv.csv'), index=False)
        
        # Grid search results
        if model.grid_search_results is not None:
            cv_results_df = pd.DataFrame(model.grid_search_results.cv_results_)
            cv_results_df.to_csv(os.path.join(self.results_dir, 'Tables', 'grid_search_results.csv'), index=False)
        
        # Best parameters
        with open(os.path.join(self.results_dir, 'Reports', 'best_params.json'), 'w') as f:
            json.dump(model.best_params, f, indent=2)
        
        # Permutation importance
        importance_df = model.get_feature_importance(method='permutation')
        if not importance_df.empty:
            importance_df.to_csv(os.path.join(self.results_dir, 'Tables', 'permutation_importance.csv'), index=False)
        
        # Reproducibility
        if model.reproducibility_info:
            with open(os.path.join(self.results_dir, 'Reports', 'reproducibility.json'), 'w') as f:
                json.dump(model.reproducibility_info, f, indent=2)
        
        # Publication summary
        report_path = os.path.join(self.results_dir, 'Reports', 'paper_summary.txt')
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("=" * 80 + "\n")
            f.write("ADABOOST MODEL - PUBLICATION READY RESULTS\n")
            f.write("CHLORIDE INGRESS - TOTAL CHLORIDE PREDICTION\n")
            f.write("VERSION 4.0 - REPEATED CV + PERMUTATION IMPORTANCE\n")
            f.write("=" * 80 + "\n\n")
            
            f.write("DATA SPLIT:\n")
            f.write("-" * 40 + "\n")
            f.write(f"  Training: {model.n_train_samples} samples (80%)\n")
            f.write(f"  Testing : {model.n_test_samples} samples (20%)\n")
            f.write(f"  Features: {model.n_features}\n")
            
            f.write("\nCROSS-VALIDATION (RepeatedKFold 5×3):\n")
            f.write("-" * 40 + "\n")
            f.write(f"  R²  : {model.metrics_cv['R2_stats']['mean']:.4f} ± {model.metrics_cv['R2_stats']['std']:.4f}\n")
            f.write(f"  R² 95% CI: [{model.metrics_cv['R2_stats']['ci_lower']:.4f}, {model.metrics_cv['R2_stats']['ci_upper']:.4f}]\n")
            f.write(f"  RMSE: {model.metrics_cv['RMSE_stats']['mean']:.4f} ± {model.metrics_cv['RMSE_stats']['std']:.4f}\n")
            f.write(f"  RMSE 95% CI: [{model.metrics_cv['RMSE_stats']['ci_lower']:.4f}, {model.metrics_cv['RMSE_stats']['ci_upper']:.4f}]\n")
            f.write(f"  Number of folds: {len(model.metrics_cv['fold_metrics'])}\n")
            
            f.write("\nTEST SET METRICS:\n")
            f.write("-" * 40 + "\n")
            for metric, value in model.metrics_test.items():
                f.write(f"  {metric:15s}: {value:.4f}\n")
            
            f.write("\nBOOTSTRAP 95% CI (Test Set):\n")
            f.write("-" * 40 + "\n")
            for metric, values in model.metrics_test_bootstrap.items():
                f.write(f"  {metric:15s}: {values['mean']:.4f} [{values['lower_95']:.4f}, {values['upper_95']:.4f}]\n")
            
            f.write("\nBOOTSTRAP 95% CI (Cross-Validation):\n")
            f.write("-" * 40 + "\n")
            for metric, values in model.metrics_cv_bootstrap.items():
                f.write(f"  {metric:15s}: {values['mean']:.4f} [{values['lower_95']:.4f}, {values['upper_95']:.4f}]\n")
            
            f.write("\nBEST HYPERPARAMETERS:\n")
            f.write("-" * 40 + "\n")
            for param, value in model.best_params.items():
                clean_param = param.replace('model__estimator__', 'base_').replace('model__', '')
                f.write(f"  {clean_param:25s}: {value}\n")
            
            f.write("\nPERMUTATION IMPORTANCE (Top 10):\n")
            f.write("-" * 40 + "\n")
            importance_df = model.get_feature_importance(method='permutation')
            if not importance_df.empty:
                for i, row in importance_df.head(10).iterrows():
                    f.write(f"  {row['Feature']:30s}: {row['Importance_mean']:.4f} ± {row['Importance_std']:.4f}\n")
            
            f.write("\nRESIDUAL ANALYSIS (Test Set):\n")
            f.write("-" * 40 + "\n")
            for metric, value in model.residual_analysis['test'].items():
                if metric != 'shapiro_wilk_p' or not np.isnan(value):
                    f.write(f"  {metric:20s}: {value:.4f}\n")
            
            f.write("\n" + "=" * 80 + "\n")
        
        logger.info(f"All results saved to: {self.results_dir}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    print("\n" + "=" * 80)
    print("SCIENTIFIC ADABOOST - TOTAL CHLORIDE PREDICTION")
    print("VERSION 4.0 - REPEATED CV + PERMUTATION IMPORTANCE")
    print("=" * 80)
    print("\nHYPERPARAMETER GRID:")
    print("  n_estimators    : [50, 100, 200]")
    print("  learning_rate   : [0.05, 0.1, 0.2]")
    print("  max_depth       : [2, 3, 4]")
    print("  min_samples_split: [2, 5]")
    print("  min_samples_leaf: [1, 2]")
    print("  loss            : ['linear', 'square']")
    print("  Total combinations: 216")
    print("\nCROSS-VALIDATION:")
    print("  Method: RepeatedKFold")
    print("  Splits: 5")
    print("  Repeats: 3")
    print("  Total folds: 15")
    print("  Total CV fits: 216 × 15 = 3,240")
    print("\nFEATURE IMPORTANCE:")
    print("  Method: Permutation Importance")
    print("  Repeats: 30")
    print("\n" + "=" * 80 + "\n")
    
    try:
        data_path = r"D:\2026 Work\My Papers\Chloride Ingress\Data\Data.csv"
        
        trainer = ScientificModelTrainer()
        model = trainer.run_pipeline(data_path)
        
        print("\n" + "=" * 80)
        print("FINAL RESULTS SUMMARY")
        print("=" * 80)
        
        print(f"\nData Split:")
        print(f"  Training: {model.n_train_samples} samples (80%)")
        print(f"  Testing : {model.n_test_samples} samples (20%)")
        print(f"  Features: {model.n_features}")
        
        print(f"\nCross-Validation (RepeatedKFold 5×3 = 15 folds):")
        print(f"  R²  : {model.metrics_cv['R2_stats']['mean']:.4f} ± {model.metrics_cv['R2_stats']['std']:.4f}")
        print(f"  R² 95% CI: [{model.metrics_cv['R2_stats']['ci_lower']:.4f}, {model.metrics_cv['R2_stats']['ci_upper']:.4f}]")
        print(f"  RMSE: {model.metrics_cv['RMSE_stats']['mean']:.4f} ± {model.metrics_cv['RMSE_stats']['std']:.4f}")
        print(f"  RMSE 95% CI: [{model.metrics_cv['RMSE_stats']['ci_lower']:.4f}, {model.metrics_cv['RMSE_stats']['ci_upper']:.4f}]")
        
        print(f"\nTest Set Performance:")
        for metric, value in model.metrics_test.items():
            print(f"  {metric:15s}: {value:.4f}")
        
        print(f"\nBootstrap 95% CI (Test Set - R²):")
        print(f"  {model.metrics_test_bootstrap['R2']['mean']:.4f} "
              f"[{model.metrics_test_bootstrap['R2']['lower_95']:.4f}, "
              f"{model.metrics_test_bootstrap['R2']['upper_95']:.4f}]")
        
        print(f"\nBootstrap 95% CI (CV - R²):")
        print(f"  {model.metrics_cv_bootstrap['R2']['mean']:.4f} "
              f"[{model.metrics_cv_bootstrap['R2']['lower_95']:.4f}, "
              f"{model.metrics_cv_bootstrap['R2']['upper_95']:.4f}]")
        
        print(f"\nBest Parameters:")
        for param, value in model.best_params.items():
            clean_param = param.replace('model__estimator__', 'base_').replace('model__', '')
            print(f"  {clean_param:25s}: {value}")
        
        print(f"\nPermutation Importance (Top 5):")
        importance_df = model.get_feature_importance(method='permutation')
        if not importance_df.empty:
            for i, row in importance_df.head(5).iterrows():
                print(f"  {row['Feature']:30s}: {row['Importance_mean']:.4f} ± {row['Importance_std']:.4f}")
        
        print(f"\nResults saved to: {trainer.results_dir}")
        print("=" * 80)
        
    except Exception as e:
        logger.error(f"Training failed: {e}")
        import traceback
        traceback.print_exc()
        raise

if __name__ == "__main__":
    main() 

2026-08-05 12:29:41,432 - INFO - ================================================================================
2026-08-05 12:29:41,435 - INFO - SCIENTIFIC ADABOOST - TOTAL CHLORIDE PREDICTION
2026-08-05 12:29:41,436 - INFO - VERSION 4.0 - REPEATED CV + PERMUTATION IMPORTANCE
2026-08-05 12:29:41,437 - INFO - ================================================================================
2026-08-05 12:29:41,438 - INFO - 
HYPERPARAMETER GRID:
2026-08-05 12:29:41,440 - INFO -   n_estimators    : [50, 100, 200]
2026-08-05 12:29:41,442 - INFO -   learning_rate   : [0.05, 0.1, 0.2]
2026-08-05 12:29:41,443 - INFO -   max_depth       : [2, 3, 4]
2026-08-05 12:29:41,445 - INFO -   min_samples_split: [2, 5]
2026-08-05 12:29:41,446 - INFO -   min_samples_leaf: [1, 2]
2026-08-05 12:29:41,448 - INFO -   loss            : ['linear', 'square']
2026-08-05 12:29:41,450 - INFO -   Total combinations: 216
2026-08-05 12:29:41,452 - INFO -   CV: RepeatedKFold (5×3 = 15 splits)
2026-08-05 12:29:41,453 - 


SCIENTIFIC ADABOOST - TOTAL CHLORIDE PREDICTION
VERSION 4.0 - REPEATED CV + PERMUTATION IMPORTANCE

HYPERPARAMETER GRID:
  n_estimators    : [50, 100, 200]
  learning_rate   : [0.05, 0.1, 0.2]
  max_depth       : [2, 3, 4]
  min_samples_split: [2, 5]
  min_samples_leaf: [1, 2]
  loss            : ['linear', 'square']
  Total combinations: 216

CROSS-VALIDATION:
  Method: RepeatedKFold
  Splits: 5
  Repeats: 3
  Total folds: 15
  Total CV fits: 216 × 15 = 3,240

FEATURE IMPORTANCE:
  Method: Permutation Importance
  Repeats: 30




2026-08-05 12:29:43,191 - INFO - Initial data shape: (918, 16)
2026-08-05 12:29:43,225 - INFO - Loaded 918 samples with 15 features
2026-08-05 12:29:43,231 - INFO - Target stats - Min: 0.0000, Max: 9.3500, Mean: 1.6777
2026-08-05 12:29:43,236 - INFO - ================================================================================
2026-08-05 12:29:43,237 - INFO - SCIENTIFIC ADABOOST - 80/20 SPLIT WITH REPEATED CV
2026-08-05 12:29:43,238 - INFO - ================================================================================
2026-08-05 12:29:43,271 - INFO - Data Split (80/20):
2026-08-05 12:29:43,272 - INFO -   Training: 734 samples (80.0%)
2026-08-05 12:29:43,273 - INFO -   Testing : 184 samples (20.0%)
2026-08-05 12:29:43,276 - INFO - 
GridSearchCV with RepeatedKFold (5×3 = 15 splits):
2026-08-05 12:29:43,278 - INFO -   Parameter combinations: 216
2026-08-05 12:29:43,279 - INFO -   Total fits: 3240
2026-08-05 12:29:43,281 - INFO -   Estimated time: ~64.8 seconds (with n_jobs=-1)
2026


FINAL RESULTS SUMMARY

Data Split:
  Training: 734 samples (80%)
  Testing : 184 samples (20%)
  Features: 15

Cross-Validation (RepeatedKFold 5×3 = 15 folds):
  R²  : 0.7441 ± 0.0359
  R² 95% CI: [0.6995, 0.7887]
  RMSE: 0.6756 ± 0.0616
  RMSE 95% CI: [0.5991, 0.7522]

Test Set Performance:
  R2             : 0.7866
  RMSE           : 0.6056
  MAE            : 0.4991
  MAPE           : 4.4924
  NRMSE_mean     : 0.3637
  NRMSE_range    : 0.0988
  Pearson        : 0.9005
  Spearman       : 0.9130

Bootstrap 95% CI (Test Set - R²):
  0.7848 [0.7382, 0.8260]

Bootstrap 95% CI (CV - R²):
  0.7479 [0.7139, 0.7803]

Best Parameters:
  base_max_depth           : 4
  base_min_samples_leaf    : 2
  base_min_samples_split   : 2
  learning_rate            : 0.2
  loss                     : square
  n_estimators             : 200

Permutation Importance (Top 5):
  Depth_from_surface_mm         : 1.1750 ± 0.1042
  Age__year                     : 0.2990 ± 0.0377
  Surface_chloride_g_L          : 0.

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
import os
import joblib
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# ENHANCED STYLING CONFIGURATION
# ============================================================================
# Set publication-quality style with enhanced aesthetics
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 3.0
plt.rcParams['lines.linewidth'] = 3.0
plt.rcParams['xtick.major.width'] = 3.0
plt.rcParams['ytick.major.width'] = 3.0
plt.rcParams['xtick.minor.width'] = 2.0
plt.rcParams['ytick.minor.width'] = 2.0
plt.rcParams['legend.framealpha'] = 0.95
plt.rcParams['legend.edgecolor'] = 'black'
plt.rcParams['legend.fancybox'] = False
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#FAFAFA'
plt.rcParams['figure.facecolor'] = 'white'

# Enhanced color palette with premium colors
COLORS = {
    'training': '#1A5276',      # Deep navy blue
    'training_light': '#2E86C1',
    'testing': '#922B21',       # Deep burgundy red
    'testing_light': '#E74C3C',
    'perfect': '#000000',       # Black
    'trend': '#D35400',         # Deep orange
    'trend_light': '#E67E22',
    'zero_line': '#2C3E50',     # Dark grey
    'band_training': '#1A5276',
    'band_testing': '#922B21',
    'band_alpha': 0.12,
    'grid': '#E8E8E8',
    'grid_major': '#D0D0D0',
    'background': '#FFFFFF',
    'highlight': '#F39C12',
    'shadow': 'rgba(0,0,0,0.1)'
}

# ============================================================================
# LOAD MODEL AND DATA
# ============================================================================
def load_model_and_data(model_path, data_path=None):
    """Load saved model and data."""
    model_data = joblib.load(model_path)
    pipeline = model_data['pipeline']
    results_dir = os.path.dirname(os.path.dirname(model_path))
    
    X_train = model_data.get('X_train')
    X_test = model_data.get('X_test')
    y_train = model_data.get('y_train')
    y_test = model_data.get('y_test')
    
    if X_train is None or X_test is None:
        print("Data not found in model. Loading from CSV...")
        if data_path is None:
            possible_paths = [
                r"D:\2026 Work\My Papers\Chloride Ingress\Data\Data.csv",
                os.path.join(os.path.dirname(results_dir), 'Data', 'Data.csv'),
                os.path.join(os.path.dirname(os.path.dirname(results_dir)), 'Data', 'Data.csv')
            ]
            for path in possible_paths:
                if os.path.exists(path):
                    data_path = path
                    break
        
        if data_path and os.path.exists(data_path):
            df = pd.read_csv(data_path, encoding='ISO-8859-1')
            df.columns = df.columns.str.strip()
            target_col = 'Total chloride, % of binder mass'
            
            features_to_drop = ['Exposure_zones', 'Exposure Zones', 'ExposureZone', 'exposure_zones']
            for feature in features_to_drop:
                if feature in df.columns:
                    df = df.drop(columns=[feature])
            
            X = df.drop(columns=[target_col])
            y = df[target_col]
            valid_idx = ~y.isna()
            X = X[valid_idx]
            y = y[valid_idx]
            
            import re
            def clean_name(name):
                name = re.sub(r'[()\[\]{}<>/\\|;:,.\'"]+', '_', str(name))
                name = re.sub(r'\s+', '_', name)
                name = re.sub(r'[^a-zA-Z0-9_]', '', name)
                if name and name[0].isdigit():
                    name = 'f_' + name
                if not name:
                    name = 'feature'
                return name
            X.columns = [clean_name(col) for col in X.columns]
            
            test_size = model_data.get('config', {}).get('test_size', 0.20)
            random_state = model_data.get('config', {}).get('random_state', 42)
            
            from sklearn.model_selection import train_test_split
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, random_state=random_state, shuffle=True
            )
            print(f"Loaded data from CSV: {len(X_train)} training, {len(X_test)} test samples")
        else:
            raise ValueError("Could not find data. Please provide data_path.")
    
    return {
        'pipeline': pipeline,
        'model_data': model_data,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'results_dir': results_dir
    }

# ============================================================================
# CREATE ENHANCED PREDICTION TABLES
# ============================================================================
def create_enhanced_prediction_tables(pipeline, X_train, y_train, X_test, y_test, results_dir):
    """Create enhanced prediction tables with actual, predicted, and errors."""
    
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    y_train = np.array(y_train).flatten()
    y_test = np.array(y_test).flatten()
    y_train_pred = np.array(y_train_pred).flatten()
    y_test_pred = np.array(y_test_pred).flatten()
    
    # Training table with enhanced formatting
    train_table = pd.DataFrame({
        'Sample': range(1, len(y_train) + 1),
        'Actual': y_train,
        'Predicted': y_train_pred,
        'Error': y_train - y_train_pred,
        'Absolute_Error': np.abs(y_train - y_train_pred),
        'Percentage_Error': np.abs(y_train - y_train_pred) / (np.abs(y_train) + 1e-10) * 100
    })
    
    # Test table with enhanced formatting
    test_table = pd.DataFrame({
        'Sample': range(1, len(y_test) + 1),
        'Actual': y_test,
        'Predicted': y_test_pred,
        'Error': y_test - y_test_pred,
        'Absolute_Error': np.abs(y_test - y_test_pred),
        'Percentage_Error': np.abs(y_test - y_test_pred) / (np.abs(y_test) + 1e-10) * 100
    })
    
    # Enhanced summary statistics
    summary_stats = pd.DataFrame({
        'Metric': ['Count', 'Mean_Actual', 'Mean_Predicted', 'Mean_Error', 'Mean_Abs_Error', 
                   'Mean_Pct_Error', 'Std_Actual', 'Std_Predicted', 'Std_Error', 
                   'Min_Error', 'Max_Error', 'R2', 'RMSE', 'MAE', 'MAPE'],
        'Training': [
            len(train_table),
            train_table['Actual'].mean(),
            train_table['Predicted'].mean(),
            train_table['Error'].mean(),
            train_table['Absolute_Error'].mean(),
            train_table['Percentage_Error'].mean(),
            train_table['Actual'].std(),
            train_table['Predicted'].std(),
            train_table['Error'].std(),
            train_table['Error'].min(),
            train_table['Error'].max(),
            r2_score(train_table['Actual'], train_table['Predicted']),
            np.sqrt(mean_squared_error(train_table['Actual'], train_table['Predicted'])),
            mean_absolute_error(train_table['Actual'], train_table['Predicted']),
            train_table['Percentage_Error'].mean()
        ],
        'Testing': [
            len(test_table),
            test_table['Actual'].mean(),
            test_table['Predicted'].mean(),
            test_table['Error'].mean(),
            test_table['Absolute_Error'].mean(),
            test_table['Percentage_Error'].mean(),
            test_table['Actual'].std(),
            test_table['Predicted'].std(),
            test_table['Error'].std(),
            test_table['Error'].min(),
            test_table['Error'].max(),
            r2_score(test_table['Actual'], test_table['Predicted']),
            np.sqrt(mean_squared_error(test_table['Actual'], test_table['Predicted'])),
            mean_absolute_error(test_table['Actual'], test_table['Predicted']),
            test_table['Percentage_Error'].mean()
        ]
    })
    
    # Save tables
    os.makedirs(os.path.join(results_dir, 'Tables'), exist_ok=True)
    train_table.to_csv(os.path.join(results_dir, 'Tables', 'training_predictions_enhanced.csv'), index=False)
    test_table.to_csv(os.path.join(results_dir, 'Tables', 'test_predictions_enhanced.csv'), index=False)
    summary_stats.to_csv(os.path.join(results_dir, 'Tables', 'prediction_summary_stats_enhanced.csv'), index=False)
    
    print(f"\n✅ Enhanced Prediction Tables Created:")
    print(f"  📊 Training samples: {len(train_table)}")
    print(f"  📊 Test samples: {len(test_table)}")
    print(f"  📁 Tables saved to: {os.path.join(results_dir, 'Tables')}")
    
    return train_table, test_table, summary_stats

# ============================================================================
# PLOT: ENHANCED TRAINING SCATTER
# ============================================================================
def plot_enhanced_training_scatter(y_train, y_train_pred, results_dir):
    """Plot enhanced training scatter with premium styling."""
    
    fig, ax = plt.subplots(figsize=(11, 10), facecolor='white')
    fig.patch.set_facecolor('white')
    
    # Scatter with gradient effect
    scatter = ax.scatter(y_train, y_train_pred, 
                         c=y_train_pred, cmap='Blues', s=80, alpha=0.7,
                         edgecolors='black', linewidth=2, zorder=5)
    
    # Perfect prediction line
    min_val = min(y_train.min(), y_train_pred.min()) - 0.5
    max_val = max(y_train.max(), y_train_pred.max()) + 0.5
    ax.plot([min_val, max_val], [min_val, max_val], 
            'k-', linewidth=4, label='Perfect Prediction', alpha=0.9, zorder=4)
    
    # Trend line with confidence interval
    slope, intercept, r_value, p_value, std_err = stats.linregress(y_train, y_train_pred)
    x_line = np.linspace(min_val, max_val, 100)
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, color=COLORS['trend'], linewidth=3.5, 
            label=f'Regression: y = {slope:.3f}x + {intercept:.3f}', zorder=6)
    
    # Confidence interval bands
    pred_std = np.std(y_train_pred - y_train)
    ax.fill_between(x_line, y_line - 1.96*pred_std, y_line + 1.96*pred_std,
                    alpha=0.15, color=COLORS['trend_light'], label='95% CI')
    
    # Metrics
    r2 = r2_score(y_train, y_train_pred)
    rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    mae = mean_absolute_error(y_train, y_train_pred)
    
    stats_text = (
        f'TRAINING SET\n'
        f'┌─────────────────────┐\n'
        f'│ N = {len(y_train):>6}              │\n'
        f'│ R² = {r2:>8.4f}          │\n'
        f'│ RMSE = {rmse:>7.4f}        │\n'
        f'│ MAE = {mae:>8.4f}          │\n'
        f'│ Pearson r = {r_value:>6.4f}     │\n'
        f'└─────────────────────┘'
    )
    props = dict(boxstyle='round,pad=0.8', facecolor='white', 
                 edgecolor='black', linewidth=3, alpha=0.95)
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', bbox=props, fontweight='bold', fontfamily='monospace')
    
    ax.set_xlabel('Actual Total Chloride (% of binder mass)', fontsize=15, fontweight='bold')
    ax.set_ylabel('Predicted Total Chloride (% of binder mass)', fontsize=15, fontweight='bold')
    ax.set_title('Training Set: Actual vs Predicted', fontsize=18, fontweight='bold', pad=20, color='#1A5276')
    
    # Enhanced borders
    for spine in ax.spines.values():
        spine.set_linewidth(4)
        spine.set_color('black')
    
    ax.tick_params(width=3, length=10, labelsize=13)
    ax.tick_params(which='minor', width=2, length=6)
    
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=1, color=COLORS['grid_major'])
    ax.set_axisbelow(True)
    ax.legend(loc='lower right', fontsize=12, framealpha=0.95, 
              edgecolor='black', fancybox=False, frameon=True, shadow=True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'Plots', 'Enhanced_Training_Scatter.png'), 
                dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print("  ✅ Enhanced training scatter plot saved")

# ============================================================================
# PLOT: ENHANCED TEST SCATTER
# ============================================================================
def plot_enhanced_test_scatter(y_test, y_test_pred, results_dir):
    """Plot enhanced test scatter with premium styling."""
    
    fig, ax = plt.subplots(figsize=(11, 10), facecolor='white')
    fig.patch.set_facecolor('white')
    
    # Scatter with gradient effect
    scatter = ax.scatter(y_test, y_test_pred, 
                         c=y_test_pred, cmap='Reds', s=80, alpha=0.7,
                         edgecolors='black', linewidth=2, zorder=5)
    
    min_val = min(y_test.min(), y_test_pred.min()) - 0.5
    max_val = max(y_test.max(), y_test_pred.max()) + 0.5
    ax.plot([min_val, max_val], [min_val, max_val], 
            'k-', linewidth=4, label='Perfect Prediction', alpha=0.9, zorder=4)
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(y_test, y_test_pred)
    x_line = np.linspace(min_val, max_val, 100)
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, color=COLORS['trend'], linewidth=3.5, 
            label=f'Regression: y = {slope:.3f}x + {intercept:.3f}', zorder=6)
    
    pred_std = np.std(y_test_pred - y_test)
    ax.fill_between(x_line, y_line - 1.96*pred_std, y_line + 1.96*pred_std,
                    alpha=0.15, color=COLORS['trend_light'], label='95% CI')
    
    r2 = r2_score(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    mae = mean_absolute_error(y_test, y_test_pred)
    
    stats_text = (
        f'TEST SET\n'
        f'┌─────────────────────┐\n'
        f'│ N = {len(y_test):>6}              │\n'
        f'│ R² = {r2:>8.4f}          │\n'
        f'│ RMSE = {rmse:>7.4f}        │\n'
        f'│ MAE = {mae:>8.4f}          │\n'
        f'│ Pearson r = {r_value:>6.4f}     │\n'
        f'└─────────────────────┘'
    )
    props = dict(boxstyle='round,pad=0.8', facecolor='white', 
                 edgecolor='black', linewidth=3, alpha=0.95)
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', bbox=props, fontweight='bold', fontfamily='monospace')
    
    ax.set_xlabel('Actual Total Chloride (% of binder mass)', fontsize=15, fontweight='bold')
    ax.set_ylabel('Predicted Total Chloride (% of binder mass)', fontsize=15, fontweight='bold')
    ax.set_title('Test Set: Actual vs Predicted', fontsize=18, fontweight='bold', pad=20, color='#922B21')
    
    for spine in ax.spines.values():
        spine.set_linewidth(4)
        spine.set_color('black')
    
    ax.tick_params(width=3, length=10, labelsize=13)
    ax.tick_params(which='minor', width=2, length=6)
    
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=1, color=COLORS['grid_major'])
    ax.set_axisbelow(True)
    ax.legend(loc='lower right', fontsize=12, framealpha=0.95, 
              edgecolor='black', fancybox=False, frameon=True, shadow=True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'Plots', 'Enhanced_Test_Scatter.png'), 
                dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print("  ✅ Enhanced test scatter plot saved")

# ============================================================================
# PLOT: ENHANCED COMBINED SCATTER
# ============================================================================
def plot_enhanced_combined_scatter(y_train, y_train_pred, y_test, y_test_pred, results_dir):
    """Plot enhanced combined scatter with premium styling."""
    
    fig, ax = plt.subplots(figsize=(13, 11), facecolor='white')
    fig.patch.set_facecolor('white')
    
    # Training with gradient
    scatter1 = ax.scatter(y_train, y_train_pred, 
                         c=y_train_pred, cmap='Blues', s=70, alpha=0.6,
                         edgecolors='black', linewidth=2, label='Training', zorder=3)
    
    # Testing with gradient
    scatter2 = ax.scatter(y_test, y_test_pred, 
                         c=y_test_pred, cmap='Reds', s=70, alpha=0.7,
                         edgecolors='black', linewidth=2, label='Testing', zorder=4)
    
    all_y = np.concatenate([y_train, y_test])
    all_pred = np.concatenate([y_train_pred, y_test_pred])
    min_val = min(all_y.min(), all_pred.min()) - 0.5
    max_val = max(all_y.max(), all_pred.max()) + 0.5
    ax.plot([min_val, max_val], [min_val, max_val], 
            'k-', linewidth=4, label='Perfect Prediction', alpha=0.9, zorder=5)
    
    r2_train = r2_score(y_train, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    r2_test = r2_score(y_test, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    stats_text = (
        f'Training: R² = {r2_train:.4f}, RMSE = {rmse_train:.4f}\n'
        f'Testing : R² = {r2_test:.4f}, RMSE = {rmse_test:.4f}'
    )
    props = dict(boxstyle='round,pad=0.8', facecolor='white', 
                 edgecolor='black', linewidth=3, alpha=0.95)
    ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=13,
            verticalalignment='top', bbox=props, fontweight='bold')
    
    ax.set_xlabel('Actual Total Chloride (% of binder mass)', fontsize=16, fontweight='bold')
    ax.set_ylabel('Predicted Total Chloride (% of binder mass)', fontsize=16, fontweight='bold')
    ax.set_title('Combined Training and Test Set Predictions', fontsize=19, fontweight='bold', pad=25)
    
    for spine in ax.spines.values():
        spine.set_linewidth(4)
        spine.set_color('black')
    
    ax.tick_params(width=3, length=10, labelsize=13)
    ax.tick_params(which='minor', width=2, length=6)
    
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=1, color=COLORS['grid_major'])
    ax.set_axisbelow(True)
    ax.legend(fontsize=13, framealpha=0.95, edgecolor='black', 
              fancybox=False, frameon=True, shadow=True, loc='lower right')
    
    # Add colorbars
    cbar1 = plt.colorbar(scatter1, ax=ax, location='left', pad=0.1, fraction=0.02)
    cbar1.set_label('Training Prediction Value', fontsize=10)
    cbar2 = plt.colorbar(scatter2, ax=ax, location='right', pad=0.1, fraction=0.02)
    cbar2.set_label('Test Prediction Value', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'Plots', 'Enhanced_Combined_Scatter.png'), 
                dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print("  ✅ Enhanced combined scatter plot saved")

# ============================================================================
# PLOT: ENHANCED TRAINING RESIDUALS
# ============================================================================
def plot_enhanced_training_residuals(y_train, y_train_pred, results_dir):
    """Plot enhanced training residuals with premium styling."""
    
    residuals = y_train - y_train_pred
    fitted = y_train_pred
    
    fig, ax = plt.subplots(figsize=(14, 8), facecolor='white')
    fig.patch.set_facecolor('white')
    
    # Scatter with gradient
    scatter = ax.scatter(fitted, residuals, 
                         c=residuals, cmap='coolwarm', s=60, alpha=0.7,
                         edgecolors='black', linewidth=2, zorder=5)
    
    # Zero line
    ax.axhline(y=0, color='black', linestyle='-', linewidth=4, alpha=0.9, zorder=3)
    
    # Confidence bands with gradient fill
    std_res = np.std(residuals)
    x_range = np.linspace(fitted.min(), fitted.max(), 100)
    ax.fill_between(x_range, -2*std_res, 2*std_res, 
                    alpha=0.15, color=COLORS['band_training'], 
                    label='±2σ Band', zorder=1)
    ax.fill_between(x_range, -3*std_res, 3*std_res, 
                    alpha=0.08, color=COLORS['band_training'], 
                    label='±3σ Band', zorder=0)
    ax.axhline(y=2*std_res, color=COLORS['training'], linestyle='--', 
               linewidth=3, alpha=0.8, zorder=2)
    ax.axhline(y=-2*std_res, color=COLORS['training'], linestyle='--', 
               linewidth=3, alpha=0.8, zorder=2)
    
    # LOESS-like trend
    from scipy.interpolate import UnivariateSpline
    try:
        spline = UnivariateSpline(fitted, residuals, s=0.5)
        x_smooth = np.linspace(fitted.min(), fitted.max(), 100)
        y_smooth = spline(x_smooth)
        ax.plot(x_smooth, y_smooth, color=COLORS['trend'], linewidth=3, 
                label='Smooth Trend', zorder=6)
    except:
        pass
    
    mean_res = np.mean(residuals)
    skew = stats.skew(residuals)
    kurtosis = stats.kurtosis(residuals)
    
    stats_text = (
        f'TRAINING SET RESIDUALS\n'
        f'┌──────────────────────────┐\n'
        f'│ Mean = {mean_res:>10.4f}    │\n'
        f'│ Std = {std_res:>11.4f}      │\n'
        f'│ Skew = {skew:>10.4f}      │\n'
        f'│ Kurtosis = {kurtosis:>7.4f}  │\n'
        f'└──────────────────────────┘'
    )
    props = dict(boxstyle='round,pad=0.8', facecolor='white', 
                 edgecolor='black', linewidth=3, alpha=0.95)
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', ha='right', bbox=props, fontweight='bold', fontfamily='monospace')
    
    ax.set_xlabel('Predicted Total Chloride (% of binder mass)', fontsize=15, fontweight='bold')
    ax.set_ylabel('Residual (Actual - Predicted)', fontsize=15, fontweight='bold')
    ax.set_title('Training Set: Enhanced Residual Analysis', fontsize=18, fontweight='bold', pad=20, color='#1A5276')
    
    for spine in ax.spines.values():
        spine.set_linewidth(4)
        spine.set_color('black')
    
    ax.tick_params(width=3, length=10, labelsize=13)
    ax.tick_params(which='minor', width=2, length=6)
    
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=1, color=COLORS['grid_major'])
    ax.set_axisbelow(True)
    ax.legend(loc='upper right', fontsize=12, framealpha=0.95, 
              edgecolor='black', fancybox=False, frameon=True, shadow=True)
    
    # Colorbar
    cbar = plt.colorbar(scatter, ax=ax, location='right', pad=0.05, fraction=0.03)
    cbar.set_label('Residual Value', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'Plots', 'Enhanced_Training_Residuals.png'), 
                dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print("  ✅ Enhanced training residual plot saved")

# ============================================================================
# PLOT: ENHANCED TEST RESIDUALS
# ============================================================================
def plot_enhanced_test_residuals(y_test, y_test_pred, results_dir):
    """Plot enhanced test residuals with premium styling."""
    
    residuals = y_test - y_test_pred
    fitted = y_test_pred
    
    fig, ax = plt.subplots(figsize=(14, 8), facecolor='white')
    fig.patch.set_facecolor('white')
    
    scatter = ax.scatter(fitted, residuals, 
                         c=residuals, cmap='coolwarm', s=60, alpha=0.7,
                         edgecolors='black', linewidth=2, zorder=5)
    
    ax.axhline(y=0, color='black', linestyle='-', linewidth=4, alpha=0.9, zorder=3)
    
    std_res = np.std(residuals)
    x_range = np.linspace(fitted.min(), fitted.max(), 100)
    ax.fill_between(x_range, -2*std_res, 2*std_res, 
                    alpha=0.15, color=COLORS['band_testing'], 
                    label='±2σ Band', zorder=1)
    ax.fill_between(x_range, -3*std_res, 3*std_res, 
                    alpha=0.08, color=COLORS['band_testing'], 
                    label='±3σ Band', zorder=0)
    ax.axhline(y=2*std_res, color=COLORS['testing'], linestyle='--', 
               linewidth=3, alpha=0.8, zorder=2)
    ax.axhline(y=-2*std_res, color=COLORS['testing'], linestyle='--', 
               linewidth=3, alpha=0.8, zorder=2)
    
    from scipy.interpolate import UnivariateSpline
    try:
        spline = UnivariateSpline(fitted, residuals, s=0.5)
        x_smooth = np.linspace(fitted.min(), fitted.max(), 100)
        y_smooth = spline(x_smooth)
        ax.plot(x_smooth, y_smooth, color=COLORS['trend'], linewidth=3, 
                label='Smooth Trend', zorder=6)
    except:
        pass
    
    mean_res = np.mean(residuals)
    skew = stats.skew(residuals)
    kurtosis = stats.kurtosis(residuals)
    
    stats_text = (
        f'TEST SET RESIDUALS\n'
        f'┌──────────────────────────┐\n'
        f'│ Mean = {mean_res:>10.4f}    │\n'
        f'│ Std = {std_res:>11.4f}      │\n'
        f'│ Skew = {skew:>10.4f}      │\n'
        f'│ Kurtosis = {kurtosis:>7.4f}  │\n'
        f'└──────────────────────────┘'
    )
    props = dict(boxstyle='round,pad=0.8', facecolor='white', 
                 edgecolor='black', linewidth=3, alpha=0.95)
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, fontsize=12,
            verticalalignment='top', ha='right', bbox=props, fontweight='bold', fontfamily='monospace')
    
    ax.set_xlabel('Predicted Total Chloride (% of binder mass)', fontsize=15, fontweight='bold')
    ax.set_ylabel('Residual (Actual - Predicted)', fontsize=15, fontweight='bold')
    ax.set_title('Test Set: Enhanced Residual Analysis', fontsize=18, fontweight='bold', pad=20, color='#922B21')
    
    for spine in ax.spines.values():
        spine.set_linewidth(4)
        spine.set_color('black')
    
    ax.tick_params(width=3, length=10, labelsize=13)
    ax.tick_params(which='minor', width=2, length=6)
    
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=1, color=COLORS['grid_major'])
    ax.set_axisbelow(True)
    ax.legend(loc='upper right', fontsize=12, framealpha=0.95, 
              edgecolor='black', fancybox=False, frameon=True, shadow=True)
    
    cbar = plt.colorbar(scatter, ax=ax, location='right', pad=0.05, fraction=0.03)
    cbar.set_label('Residual Value', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'Plots', 'Enhanced_Test_Residuals.png'), 
                dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print("  ✅ Enhanced test residual plot saved")

# ============================================================================
# PLOT: ENHANCED ERROR DISTRIBUTIONS
# ============================================================================
def plot_enhanced_error_distributions(y_train, y_train_pred, y_test, y_test_pred, results_dir):
    """Plot enhanced error distributions with premium styling."""
    
    train_errors = y_train - y_train_pred
    test_errors = y_test - y_test_pred
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14), facecolor='white')
    fig.patch.set_facecolor('white')
    fig.suptitle('Enhanced Error Distribution Analysis', fontsize=20, fontweight='bold', y=0.98, color='#2C3E50')
    
    # Training histogram - enhanced
    ax = axes[0, 0]
    n, bins, patches = ax.hist(train_errors, bins=30, color=COLORS['training'], 
                                alpha=0.7, edgecolor='black', linewidth=2.5)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=3.5, alpha=0.9)
    ax.axvline(x=np.mean(train_errors), color=COLORS['trend'], linestyle='--', 
               linewidth=3, alpha=0.8, label=f'Mean: {np.mean(train_errors):.4f}')
    
    # Add density curve
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(train_errors)
    x_range = np.linspace(train_errors.min(), train_errors.max(), 100)
    ax2 = ax.twinx()
    ax2.plot(x_range, kde(x_range), color=COLORS['trend_light'], linewidth=3, alpha=0.8, label='Density')
    ax2.set_ylabel('Density', fontsize=12, fontweight='bold')
    ax2.tick_params(width=2, length=6, labelsize=11)
    
    ax.set_xlabel('Residual', fontsize=14, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=14, fontweight='bold')
    ax.set_title(f'Training Residual Distribution\nMean = {np.mean(train_errors):.4f}, Std = {np.std(train_errors):.4f}', 
                fontsize=14, fontweight='bold')
    for spine in ax.spines.values():
        spine.set_linewidth(3.5)
        spine.set_color('black')
    ax.tick_params(width=3, length=8, labelsize=12)
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=1)
    ax.set_axisbelow(True)
    ax.legend(loc='upper right', fontsize=11)
    
    # Test histogram - enhanced
    ax = axes[0, 1]
    n, bins, patches = ax.hist(test_errors, bins=30, color=COLORS['testing'], 
                                alpha=0.7, edgecolor='black', linewidth=2.5)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=3.5, alpha=0.9)
    ax.axvline(x=np.mean(test_errors), color=COLORS['trend'], linestyle='--', 
               linewidth=3, alpha=0.8, label=f'Mean: {np.mean(test_errors):.4f}')
    
    kde = gaussian_kde(test_errors)
    x_range = np.linspace(test_errors.min(), test_errors.max(), 100)
    ax2 = ax.twinx()
    ax2.plot(x_range, kde(x_range), color=COLORS['trend_light'], linewidth=3, alpha=0.8, label='Density')
    ax2.set_ylabel('Density', fontsize=12, fontweight='bold')
    ax2.tick_params(width=2, length=6, labelsize=11)
    
    ax.set_xlabel('Residual', fontsize=14, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=14, fontweight='bold')
    ax.set_title(f'Test Residual Distribution\nMean = {np.mean(test_errors):.4f}, Std = {np.std(test_errors):.4f}',
                fontsize=14, fontweight='bold')
    for spine in ax.spines.values():
        spine.set_linewidth(3.5)
        spine.set_color('black')
    ax.tick_params(width=3, length=8, labelsize=12)
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=1)
    ax.set_axisbelow(True)
    ax.legend(loc='upper right', fontsize=11)
    
    # Training Q-Q - enhanced
    ax = axes[1, 0]
    stats.probplot(train_errors, dist="norm", plot=ax)
    lines = ax.get_lines()
    if len(lines) >= 2:
        lines[0].set_marker('o')
        lines[0].set_markersize(8)
        lines[0].set_markerfacecolor(COLORS['training'])
        lines[0].set_markeredgecolor('black')
        lines[0].set_markeredgewidth(2)
        lines[0].set_linestyle('None')
        lines[0].set_alpha(0.7)
        lines[1].set_linewidth(3.5)
        lines[1].set_color('black')
        lines[1].set_alpha(0.9)
    ax.set_title('Training Q-Q Plot', fontsize=14, fontweight='bold')
    ax.set_xlabel('Theoretical Quantiles', fontsize=14, fontweight='bold')
    ax.set_ylabel('Sample Quantiles', fontsize=14, fontweight='bold')
    for spine in ax.spines.values():
        spine.set_linewidth(3.5)
        spine.set_color('black')
    ax.tick_params(width=3, length=8, labelsize=12)
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=1)
    ax.set_axisbelow(True)
    
    # Add Shapiro-Wilk test result
    if len(train_errors) >= 3 and len(train_errors) <= 5000:
        _, p_value = stats.shapiro(train_errors)
        ax.text(0.05, 0.95, f'Shapiro-Wilk p = {p_value:.4f}', 
               transform=ax.transAxes, fontsize=12, fontweight='bold',
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', linewidth=2))
    
    # Test Q-Q - enhanced
    ax = axes[1, 1]
    stats.probplot(test_errors, dist="norm", plot=ax)
    lines = ax.get_lines()
    if len(lines) >= 2:
        lines[0].set_marker('o')
        lines[0].set_markersize(8)
        lines[0].set_markerfacecolor(COLORS['testing'])
        lines[0].set_markeredgecolor('black')
        lines[0].set_markeredgewidth(2)
        lines[0].set_linestyle('None')
        lines[0].set_alpha(0.7)
        lines[1].set_linewidth(3.5)
        lines[1].set_color('black')
        lines[1].set_alpha(0.9)
    ax.set_title('Test Q-Q Plot', fontsize=14, fontweight='bold')
    ax.set_xlabel('Theoretical Quantiles', fontsize=14, fontweight='bold')
    ax.set_ylabel('Sample Quantiles', fontsize=14, fontweight='bold')
    for spine in ax.spines.values():
        spine.set_linewidth(3.5)
        spine.set_color('black')
    ax.tick_params(width=3, length=8, labelsize=12)
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=1)
    ax.set_axisbelow(True)
    
    if len(test_errors) >= 3 and len(test_errors) <= 5000:
        _, p_value = stats.shapiro(test_errors)
        ax.text(0.05, 0.95, f'Shapiro-Wilk p = {p_value:.4f}', 
               transform=ax.transAxes, fontsize=12, fontweight='bold',
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', linewidth=2))
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'Plots', 'Enhanced_Error_Distributions.png'), 
                dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print("  ✅ Enhanced error distribution plots saved")

# ============================================================================
# PLOT: ENHANCED PERFORMANCE COMPARISON
# ============================================================================
def plot_enhanced_performance_comparison(y_train, y_train_pred, y_test, y_test_pred, results_dir):
    """Plot enhanced performance comparison with premium styling."""
    
    metrics = ['R²', 'RMSE', 'MAE']
    train_values = [
        r2_score(y_train, y_train_pred),
        np.sqrt(mean_squared_error(y_train, y_train_pred)),
        mean_absolute_error(y_train, y_train_pred)
    ]
    test_values = [
        r2_score(y_test, y_test_pred),
        np.sqrt(mean_squared_error(y_test, y_test_pred)),
        mean_absolute_error(y_test, y_test_pred)
    ]
    
    fig, ax = plt.subplots(figsize=(12, 8), facecolor='white')
    fig.patch.set_facecolor('white')
    
    x = np.arange(len(metrics))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, train_values, width, label='Training', 
                   color=COLORS['training'], edgecolor='black', linewidth=3)
    bars2 = ax.bar(x + width/2, test_values, width, label='Testing',
                   color=COLORS['testing'], edgecolor='black', linewidth=3)
    
    # Add value labels with black font and background
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02 * max(train_values + test_values),
                f'{height:.4f}', ha='center', va='bottom', fontsize=12, 
                fontweight='bold', color='black')
    
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02 * max(train_values + test_values),
                f'{height:.4f}', ha='center', va='bottom', fontsize=12, 
                fontweight='bold', color='black')
    
    ax.set_xlabel('Metric', fontsize=15, fontweight='bold')
    ax.set_ylabel('Value', fontsize=15, fontweight='bold')
    ax.set_title('Enhanced Performance Comparison: Training vs Testing', fontsize=18, fontweight='bold', pad=25)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=14, fontweight='bold')
    
    for spine in ax.spines.values():
        spine.set_linewidth(4)
        spine.set_color('black')
    
    ax.tick_params(width=3, length=10, labelsize=13)
    ax.tick_params(which='minor', width=2, length=6)
    
    ax.grid(True, alpha=0.2, axis='y', linestyle='-', linewidth=1, color=COLORS['grid_major'])
    ax.set_axisbelow(True)
    ax.legend(fontsize=14, framealpha=0.95, edgecolor='black', 
              fancybox=False, frameon=True, shadow=True, loc='upper right')
    
    # Add performance improvement text
    improvement_r2 = ((test_values[0] - train_values[0]) / abs(train_values[0])) * 100
    improvement_rmse = ((train_values[1] - test_values[1]) / train_values[1]) * 100
    ax.text(0.5, -0.15, f'Test vs Train: R² {improvement_r2:+.1f}%, RMSE {improvement_rmse:+.1f}%',
            transform=ax.transAxes, fontsize=13, fontweight='bold', ha='center',
            bbox=dict(boxstyle='round', facecolor='#f0f8ff', edgecolor='black', linewidth=2))
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'Plots', 'Enhanced_Performance_Comparison.png'),
                dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print("  ✅ Enhanced performance comparison plot saved")

# ============================================================================
# PLOT: ENHANCED FEATURE IMPORTANCE
# ============================================================================
def plot_enhanced_feature_importance(importance_df, results_dir):
    """Plot enhanced feature importance with premium styling."""
    
    fig, ax = plt.subplots(figsize=(14, 11), facecolor='white')
    fig.patch.set_facecolor('white')
    
    top_features = importance_df.head(15)
    
    # Enhanced color gradient
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(top_features)))[::-1]
    
    bars = ax.barh(top_features['Feature'], top_features['Importance_mean'],
                   xerr=top_features['Importance_std'] if 'Importance_std' in top_features else None,
                   color=colors, edgecolor='black', linewidth=3, capsize=6, zorder=3)
    
    # Add value labels with background
    for bar, val in zip(bars, top_features['Importance_mean']):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=12, fontweight='bold', color='black')
    
    # Add ranking numbers
    for i, (idx, row) in enumerate(top_features.iterrows()):
        ax.text(-0.01, row.name, f'#{i+1}', va='center', ha='right', 
                fontsize=11, fontweight='bold', color='black')
    
    ax.set_xlabel('Permutation Importance (R² decrease)', fontsize=15, fontweight='bold')
    ax.set_ylabel('Feature', fontsize=15, fontweight='bold')
    ax.set_title('Top 15 Feature Importances (Permutation Method)', fontsize=18, fontweight='bold', pad=25)
    
    for spine in ax.spines.values():
        spine.set_linewidth(4)
        spine.set_color('black')
    
    ax.tick_params(width=3, length=10, labelsize=13)
    ax.tick_params(which='minor', width=2, length=6)
    
    ax.grid(True, alpha=0.2, axis='x', linestyle='-', linewidth=1, color=COLORS['grid_major'])
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'Plots', 'Enhanced_Feature_Importance.png'), 
                dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    print("  ✅ Enhanced feature importance plot saved")

# ============================================================================
# CREATE ENHANCED HTML REPORT
# ============================================================================
def create_enhanced_html_report(results_dir, train_table, test_table, summary_stats):
    """Create comprehensive enhanced HTML report with premium styling."""
    
    best_params_path = os.path.join(results_dir, 'Reports', 'best_params.json')
    if os.path.exists(best_params_path):
        with open(best_params_path, 'r') as f:
            best_params = json.load(f)
    else:
        best_params = {}
    
    train_r2 = summary_stats.loc[summary_stats['Metric'] == 'R2', 'Training'].values[0]
    test_r2 = summary_stats.loc[summary_stats['Metric'] == 'R2', 'Testing'].values[0]
    train_rmse = summary_stats.loc[summary_stats['Metric'] == 'RMSE', 'Training'].values[0]
    test_rmse = summary_stats.loc[summary_stats['Metric'] == 'RMSE', 'Testing'].values[0]
    
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>Enhanced AdaBoost Model - Total Chloride Prediction Results</title>
        <style>
            * {{ margin: 0; padding: 0; box-sizing: border-box; }}
            body {{ font-family: 'Arial', sans-serif; background: linear-gradient(135deg, #e8edf5 0%, #d5dce4 100%); padding: 40px; }}
            .container {{ max-width: 1400px; margin: 0 auto; background: white; padding: 45px; border-radius: 20px; box-shadow: 0 25px 70px rgba(0,0,0,0.35); border: 4px solid black; }}
            h1 {{ color: #1A5276; border-bottom: 5px solid #1A5276; padding-bottom: 20px; font-size: 32px; }}
            h2 {{ color: #1A5276; margin-top: 40px; font-size: 24px; border-left: 6px solid #1A5276; padding-left: 20px; }}
            .metric-grid {{ display: flex; flex-wrap: wrap; gap: 20px; justify-content: center; margin: 30px 0; }}
            .metric-box {{ background: linear-gradient(135deg, #f0f8ff, #dce6f5); padding: 25px; border-radius: 15px; border: 4px solid black; min-width: 180px; text-align: center; box-shadow: 0 6px 12px rgba(0,0,0,0.15); transition: transform 0.3s; }}
            .metric-box:hover {{ transform: scale(1.05); }}
            .metric-value {{ font-size: 32px; font-weight: bold; color: #922B21; }}
            .metric-label {{ font-size: 15px; color: #333; font-weight: bold; }}
            table {{ border-collapse: collapse; width: 100%; margin: 20px 0; border: 4px solid black; }}
            th, td {{ border: 2px solid black; padding: 12px; text-align: right; }}
            th {{ background: linear-gradient(135deg, #1A5276, #2E86C1); color: white; font-weight: bold; font-size: 15px; }}
            tr:nth-child(even) {{ background-color: #f8f9fa; }}
            tr:hover {{ background-color: #dce6f5; }}
            .plot-container {{ margin: 40px 0; text-align: center; background: #fafafa; padding: 25px; border-radius: 15px; border: 4px solid black; box-shadow: 0 4px 8px rgba(0,0,0,0.1); }}
            .plot-container img {{ max-width: 100%; border: 3px solid #ddd; border-radius: 10px; }}
            .plot-container p {{ margin-top: 15px; font-weight: bold; font-size: 16px; color: #333; }}
            .note {{ background: linear-gradient(135deg, #fff3cd, #ffeaa7); padding: 20px; border-left: 8px solid #f39c12; margin: 20px 0; border: 3px solid black; border-radius: 8px; }}
            .prediction-table {{ max-height: 500px; overflow-y: auto; border: 4px solid black; border-radius: 8px; }}
            .prediction-table table {{ border: none; }}
            .prediction-table table th {{ position: sticky; top: 0; z-index: 10; }}
            ul {{ list-style: none; padding: 0; }}
            ul li {{ padding: 10px 20px; margin: 8px 0; background: #f8f9fa; border-left: 5px solid #1A5276; border: 3px solid black; border-radius: 8px; font-weight: bold; }}
            .footer {{ text-align: center; margin-top: 50px; padding-top: 25px; border-top: 4px solid black; color: #666; }}
            .badge {{ display: inline-block; background: linear-gradient(135deg, #1A5276, #2E86C1); color: white; padding: 6px 20px; border-radius: 25px; font-size: 13px; font-weight: bold; border: 2px solid black; margin: 3px; }}
            .section-title {{ background: linear-gradient(135deg, #1A5276, #2E86C1); color: white; padding: 15px 25px; border-radius: 10px; margin: 30px 0 20px 0; border: 3px solid black; }}
            .section-title h2 {{ color: white; margin: 0; border: none; padding: 0; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>🔬 Enhanced AdaBoost Model Results</h1>
        <p style="font-size: 18px; margin: 15px 0;"><strong>Total Chloride Prediction</strong> | 
        <span class="badge">AdaBoostRegressor</span> 
        <span class="badge">RepeatedKFold 5×3</span> 
        <span class="badge">GridSearchCV 216</span>
        <span class="badge">DPI 600</span></p>
        
        <div class="note">
            <strong>📌 Key Information:</strong> 80/20 train/test split | RepeatedKFold (5×3 = 15 splits) | Permutation Importance | Enhanced Aesthetic Visualizations
        </div>
        
        <div class="section-title"><h2>📊 Model Performance Summary</h2></div>
        <div class="metric-grid">
    """
    
    metrics_data = {
        'Training R²': f"{train_r2:.4f}",
        'Test R²': f"{test_r2:.4f}",
        'Training RMSE': f"{train_rmse:.4f}",
        'Test RMSE': f"{test_rmse:.4f}",
        'Training MAE': f"{summary_stats.loc[summary_stats['Metric'] == 'MAE', 'Training'].values[0]:.4f}",
        'Test MAE': f"{summary_stats.loc[summary_stats['Metric'] == 'MAE', 'Testing'].values[0]:.4f}"
    }
    
    for label, value in metrics_data.items():
        html_content += f"""
        <div class="metric-box">
            <div class="metric-label">{label}</div>
            <div class="metric-value">{value}</div>
        </div>
        """
    
    html_content += """
        </div>
        
        <div class="section-title"><h2>📈 Enhanced Scatter Plots</h2></div>
        <div class="plot-container">
            <img src="../Plots/Enhanced_Training_Scatter.png" alt="Enhanced Training Scatter">
            <p><strong>Figure 1:</strong> Enhanced Training Set: Actual vs Predicted</p>
        </div>
        <div class="plot-container">
            <img src="../Plots/Enhanced_Test_Scatter.png" alt="Enhanced Test Scatter">
            <p><strong>Figure 2:</strong> Enhanced Test Set: Actual vs Predicted</p>
        </div>
        <div class="plot-container">
            <img src="../Plots/Enhanced_Combined_Scatter.png" alt="Enhanced Combined Scatter">
            <p><strong>Figure 3:</strong> Enhanced Combined Training and Test Set Predictions</p>
        </div>
        
        <div class="section-title"><h2>📉 Enhanced Residual Analysis</h2></div>
        <div class="plot-container">
            <img src="../Plots/Enhanced_Training_Residuals.png" alt="Enhanced Training Residuals">
            <p><strong>Figure 4:</strong> Enhanced Training Set Residual Analysis</p>
        </div>
        <div class="plot-container">
            <img src="../Plots/Enhanced_Test_Residuals.png" alt="Enhanced Test Residuals">
            <p><strong>Figure 5:</strong> Enhanced Test Set Residual Analysis</p>
        </div>
        
        <div class="section-title"><h2>📊 Enhanced Error Distributions</h2></div>
        <div class="plot-container">
            <img src="../Plots/Enhanced_Error_Distributions.png" alt="Enhanced Error Distributions">
            <p><strong>Figure 6:</strong> Enhanced Error Distribution Analysis</p>
        </div>
        
        <div class="section-title"><h2>🏆 Enhanced Performance Comparison</h2></div>
        <div class="plot-container">
            <img src="../Plots/Enhanced_Performance_Comparison.png" alt="Enhanced Performance Comparison">
            <p><strong>Figure 7:</strong> Enhanced Training vs Testing Performance Comparison</p>
        </div>
        
        <div class="section-title"><h2>🔑 Enhanced Feature Importance</h2></div>
        <div class="plot-container">
            <img src="../Plots/Enhanced_Feature_Importance.png" alt="Enhanced Feature Importance">
            <p><strong>Figure 8:</strong> Enhanced Top 15 Feature Importances</p>
        </div>
        
        <div class="section-title"><h2>📋 Enhanced Prediction Tables</h2></div>
        <h3>📊 Training Set Predictions (First 20 samples)</h3>
        <div class="prediction-table">
            {train_table.head(20).to_html(classes='table', float_format='%.4f')}
        </div>
        
        <h3>📊 Test Set Predictions (All {len(test_table)} samples)</h3>
        <div class="prediction-table">
            {test_table.to_html(classes='table', float_format='%.4f')}
        </div>
        
        <div class="section-title"><h2>📊 Enhanced Summary Statistics</h2></div>
        {summary_stats.to_html(classes='table', float_format='%.4f')}
        
        <div class="section-title"><h2>⚙️ Best Hyperparameters</h2></div>
        <ul>
    """
    
    for param, value in best_params.items():
        clean_param = param.replace('model__estimator__', 'base_').replace('model__', '')
        html_content += f"<li><strong>{clean_param}:</strong> {value}</li>"
    
    html_content += f"""
        </ul>
        
        <div class="section-title"><h2>📌 Data Split Information</h2></div>
        <ul>
            <li><strong>Training samples:</strong> {len(train_table)} (80%)</li>
            <li><strong>Test samples:</strong> {len(test_table)} (20%)</li>
            <li><strong>Total samples:</strong> {len(train_table) + len(test_table)}</li>
            <li><strong>Features:</strong> {len(train_table.columns) - 5}</li>
        </ul>
        
        <div class="footer">
            <strong>Generated on:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}<br>
            <span style="color: #1A5276; font-weight: bold; font-size: 18px;">✦ Enhanced Publication-Ready Results ✦</span><br>
            <span style="color: #666; font-size: 12px;">All plots generated at 600 DPI with premium styling</span>
        </div>
    </div>
</body>
</html>
    """
    
    report_path = os.path.join(results_dir, 'Reports', 'enhanced_model_report.html')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"  ✅ Enhanced HTML report saved to: {report_path}")

# ============================================================================
# MAIN FUNCTION
# ============================================================================
def generate_enhanced_all_plots(model_path, data_path=None):
    """Generate all enhanced plots and tables with premium styling."""
    
    print("=" * 80)
    print("🌟 GENERATING ENHANCED PUBLICATION-QUALITY VISUALIZATIONS")
    print("=" * 80)
    print("  📊 DPI: 600")
    print("  🎨 Enhanced Aesthetic Design")
    print("  📈 Premium Color Schemes")
    print("  🔍 Comprehensive Tables")
    print("=" * 80)
    
    print("\n" + "=" * 50)
    print("STEP 1: Loading Model and Data")
    print("=" * 50)
    
    try:
        data = load_model_and_data(model_path, data_path)
        pipeline = data['pipeline']
        X_train = data['X_train']
        X_test = data['X_test']
        y_train = data['y_train']
        y_test = data['y_test']
        results_dir = data['results_dir']
        
        print(f"  ✅ Model loaded from: {model_path}")
        print(f"  ✅ Training samples: {len(X_train)}")
        print(f"  ✅ Test samples: {len(X_test)}")
        print(f"  ✅ Results directory: {results_dir}")
        
    except Exception as e:
        print(f"  ❌ Error loading model: {e}")
        return
    
    print("\n" + "=" * 50)
    print("STEP 2: Creating Enhanced Prediction Tables")
    print("=" * 50)
    
    train_table, test_table, summary_stats = create_enhanced_prediction_tables(
        pipeline, X_train, y_train, X_test, y_test, results_dir
    )
    
    y_train_vals = train_table['Actual'].values
    y_train_pred_vals = train_table['Predicted'].values
    y_test_vals = test_table['Actual'].values
    y_test_pred_vals = test_table['Predicted'].values
    
    print("\n" + "=" * 50)
    print("STEP 3: Generating Enhanced Visualizations")
    print("=" * 50)
    
    # Enhanced scatter plots
    plot_enhanced_training_scatter(y_train_vals, y_train_pred_vals, results_dir)
    plot_enhanced_test_scatter(y_test_vals, y_test_pred_vals, results_dir)
    plot_enhanced_combined_scatter(y_train_vals, y_train_pred_vals, y_test_vals, y_test_pred_vals, results_dir)
    
    # Enhanced residual plots
    plot_enhanced_training_residuals(y_train_vals, y_train_pred_vals, results_dir)
    plot_enhanced_test_residuals(y_test_vals, y_test_pred_vals, results_dir)
    
    # Enhanced error distributions
    plot_enhanced_error_distributions(y_train_vals, y_train_pred_vals, y_test_vals, y_test_pred_vals, results_dir)
    
    # Enhanced performance comparison
    plot_enhanced_performance_comparison(y_train_vals, y_train_pred_vals, y_test_vals, y_test_pred_vals, results_dir)
    
    print("\n" + "=" * 50)
    print("STEP 4: Generating Enhanced Feature Importance")
    print("=" * 50)
    
    importance_path = os.path.join(results_dir, 'Tables', 'permutation_importance.csv')
    if os.path.exists(importance_path):
        importance_df = pd.read_csv(importance_path)
        print(f"  ✅ Feature importance loaded from: {importance_path}")
        plot_enhanced_feature_importance(importance_df, results_dir)
    else:
        print("  ⚠️ Feature importance not found")
    
    print("\n" + "=" * 50)
    print("STEP 5: Creating Enhanced HTML Report")
    print("=" * 50)
    create_enhanced_html_report(results_dir, train_table, test_table, summary_stats)
    
    print("\n" + "=" * 80)
    print("✅ ALL ENHANCED VISUALIZATIONS GENERATED SUCCESSFULLY!")
    print("=" * 80)
    print(f"\n📁 Results saved to: {results_dir}")
    print(f"\n📊 Enhanced Files Created:")
    print(f"  📋 Tables:")
    print(f"     - training_predictions_enhanced.csv ({len(train_table)} samples)")
    print(f"     - test_predictions_enhanced.csv ({len(test_table)} samples)")
    print(f"     - prediction_summary_stats_enhanced.csv")
    print(f"  📈 Plots (DPI: 600):")
    print(f"     - Enhanced_Training_Scatter.png")
    print(f"     - Enhanced_Test_Scatter.png")
    print(f"     - Enhanced_Combined_Scatter.png")
    print(f"     - Enhanced_Training_Residuals.png")
    print(f"     - Enhanced_Test_Residuals.png")
    print(f"     - Enhanced_Error_Distributions.png")
    print(f"     - Enhanced_Performance_Comparison.png")
    print(f"     - Enhanced_Feature_Importance.png")
    print(f"  📄 Report:")
    print(f"     - enhanced_model_report.html (Premium Design)")
    print("=" * 80)

# ============================================================================
# RUN
# ============================================================================
if __name__ == "__main__":
    model_path = r"D:\2026 Work\My Papers\Chloride Ingress\Models\AdaBoost\Models\AdaBoost_80_20.joblib"
    data_path = r"D:\2026 Work\My Papers\Chloride Ingress\Data\Data.csv"
    
    if not os.path.exists(model_path):
        print(f"ERROR: Model not found at: {model_path}")
        print("Please make sure you have trained the model first.")
    else:
        generate_enhanced_all_plots(model_path, data_path)

🌟 GENERATING ENHANCED PUBLICATION-QUALITY VISUALIZATIONS
  📊 DPI: 600
  🎨 Enhanced Aesthetic Design
  📈 Premium Color Schemes
  🔍 Comprehensive Tables

STEP 1: Loading Model and Data
Data not found in model. Loading from CSV...
Loaded data from CSV: 734 training, 184 test samples
  ✅ Model loaded from: D:\2026 Work\My Papers\Chloride Ingress\Models\AdaBoost\Models\AdaBoost_80_20.joblib
  ✅ Training samples: 734
  ✅ Test samples: 184
  ✅ Results directory: D:\2026 Work\My Papers\Chloride Ingress\Models\AdaBoost

STEP 2: Creating Enhanced Prediction Tables

✅ Enhanced Prediction Tables Created:
  📊 Training samples: 734
  📊 Test samples: 184
  📁 Tables saved to: D:\2026 Work\My Papers\Chloride Ingress\Models\AdaBoost\Tables

STEP 3: Generating Enhanced Visualizations
  ✅ Enhanced training scatter plot saved
  ✅ Enhanced test scatter plot saved
  ✅ Enhanced combined scatter plot saved
  ✅ Enhanced training residual plot saved
  ✅ Enhanced test residual plot saved
  ✅ Enhanced error distri